In [1]:
import os
import re
import requests
from bs4 import BeautifulSoup
import gzip
import shutil
import pandas as pd
from datetime import datetime

# 1. Define paths and URLs
url = "http://10.24.42.15/Mps_Forward_Report/"
download_dir = r"C:\Users\mohammadali1.vc\Desktop\Python data\Large Python"
os.makedirs(download_dir, exist_ok=True)

# Dynamically get today's date in your portal's exact format: YYYY-Mon-DD (e.g., 2026-Jun-17)
today_str = datetime.today().strftime('%Y-%b-%d')
print(f"Connecting to portal... Looking for today's data ({today_str})")

response = requests.get(url)

if response.status_code == 200:
    soup = BeautifulSoup(response.text, 'html.parser')
    links = [a['href'] for a in soup.find_all('a', href=True)]
    
    # Strictly target today's date AND the 09_20 timestamp
    # Pattern looks exactly for: mps_forward_report_(Today's Date)_09_20_XX.csv or .csv.gz
    target_pattern = re.compile(rf"mps_forward_report_{today_str}_09_20_.*\.csv(?:\.gz)?")
    matching_links = [l for l in links if target_pattern.search(l)]
    
    if matching_links:
        # Get the latest entry (if there happen to be multiple seconds revisions)
        latest_file_name = sorted(matching_links)[-1]
        file_url = url + latest_file_name
        
        local_compressed_path = os.path.join(download_dir, latest_file_name)
        
        # 2. Download today's file
        print(f"Found today's correct file: {latest_file_name}")
        print("Downloading...")
        with requests.get(file_url, stream=True) as r:
            r.raise_for_status()
            with open(local_compressed_path, 'wb') as f:
                shutil.copyfileobj(r.raw, f)
        
        # 3. Extract if .gz
        if latest_file_name.endswith('.gz'):
            local_csv_path = local_compressed_path.replace('.gz', '')
            print("Extracting GZ archive...")
            with gzip.open(local_compressed_path, 'rb') as f_in:
                with open(local_csv_path, 'wb') as f_out:
                    shutil.copyfileobj(f_in, f_out)
            os.remove(local_compressed_path)
            final_file = local_csv_path
        else:
            final_file = local_compressed_path
            
        print(f"Successfully saved to: {final_file}")
        
        # 4. Load into DataFrame using the robust python engine
        os.chdir(download_dir)
        print("Parsing data file (this may take a moment for large files)...")
        
        df = pd.read_csv(
            os.path.basename(final_file), 
            engine='python', 
            on_bad_lines='skip'
        )
        
        print("Today's data loaded successfully into 'df' variable!")
        
    else:
        print(f"Could not find today's file ({today_str}) with 09:20 timestamp yet.")
else:
    print(f"Failed to access the portal. Status code: {response.status_code}")

Connecting to portal... Looking for today's data (2026-Nov-23)
Found today's correct file: mps_forward_report_2026-Nov-23_09_20_01.csv.gz
Downloading...
Extracting GZ archive...
Successfully saved to: C:\Users\mohammadali1.vc\Desktop\Python data\Large Python\mps_forward_report_2026-Nov-23_09_20_01.csv
Parsing data file (this may take a moment for large files)...
Today's data loaded successfully into 'df' variable!


In [2]:
pd.set_option("display.max_columns",None)
df

,ShipmentId,CurrentHubCity,consignmentId,ConsignmentCreationTime,ConsignmentCreationHub,parentId,MPS_count,OriginHub,CurrentHubName,DestinationHub,LastUpdateTime,LatestShipmentStatus,OrderId,CPD,ShipmentCreationTime,CustomerCity,CustomerPincode,Shipment Movement,service_type,ShipmentCategory,isMPS,isPrexo,ProductDescription,OBDType,PriorityScore,PriorityScoreV2,ItemType
0,ALWC0000000371,NaN,NaN,NaN,NaN,ALWN0000000371,0,Centralhub_AYODHYA,Centralhub_AYODHYA,SatelliteHub_JHS,2026-10-22 09:31:15,Expected,ORD-0035822,2026-11-01 09:31:06,2026-10-22 09:31:15,NaN,284201,NON-FBF,Forward,Furniture,True,NaN,Nufield White Excel Radish Hybrid 50G,NaN,0.0,P4,ext_mps
1,ALWC0000000372,NaN,NaN,NaN,NaN,ALWN0000000371,0,Centralhub_AYODHYA,Centralhub_AYODHYA,SatelliteHub_JHS,2026-10-22 09:31:15,Expected,ORD-0035822,2026-11-01 09:31:06,2026-10-22 09:31:15,NaN,284201,NON-FBF,Forward,Furniture,True,NaN,Nufield White Excel Radish Hybrid 50G,NaN,0.0,P4,ext_mps
2,ALWC0000000373,NaN,NaN,NaN,NaN,ALWN0000000371,0,Centralhub_AYODHYA,Centralhub_AYODHYA,SatelliteHub_JHS,2026-10-22 09:31:15,Expected,ORD-0035822,2026-11-01 09:31:06,2026-10-22 09:31:15,NaN,284201,NON-FBF,Forward,Furniture,True,NaN,Nufield White Excel Radish Hybrid 50G,NaN,0.0,P4,ext_mps
3,ALWC0000000374,NaN,NaN,NaN,NaN,ALWN0000000371,0,Centralhub_AYODHYA,Centralhub_AYODHYA,SatelliteHub_JHS,2026-10-22 09:31:16,Expected,ORD-0035822,2026-11-01 09:31:06,2026-10-22 09:31:15,NaN,284201,NON-FBF,Forward,Furniture,True,NaN,Nufield White Excel Radish Hybrid 50G,NaN,0.0,P4,ext_mps
4,ALWC0000000375,NaN,NaN,NaN,NaN,ALWN0000000371,0,Centralhub_AYODHYA,Centralhub_AYODHYA,SatelliteHub_JHS,2026-10-22 09:31:16,Expected,ORD-0035822,2026-11-01 09:31:06,2026-10-22 09:31:16,NaN,284201,NON-FBF,Forward,Furniture,True,NaN,Nufield White Excel Radish Hybrid 50G,NaN,0.0,P4,ext_mps
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
720580,ZZLP0000000075,NaN,NaN,NaN,NaN,ZZLN0000000079,0,Centralhub_FKHAL,Centralhub_FKHAL,SatelliteHub_KLYW,2026-11-17 15:11:53,Expected,Testorder12novT60941,2026-11-27 15:11:44,2026-11-17 15:11:53,NaN,421301,NON-FBF,Forward,Furniture,True,NaN,Vergo Bloom Dining Chair with Velvet Fabric (D...,NaN,0.0,P4,ext_mps
720581,ZZLP0000000076,NaN,NaN,NaN,NaN,ZZLN0000000079,0,Centralhub_FKHAL,Centralhub_FKHAL,SatelliteHub_KLYW,2026-11-17 15:11:53,Expected,Testorder12novT60941,2026-11-27 15:11:44,2026-11-17 15:11:53,NaN,421301,NON-FBF,Forward,Furniture,True,NaN,Vergo Bloom Dining Chair with Velvet Fabric (D...,NaN,0.0,P4,ext_mps
720582,ZZLP0000000077,NaN,NaN,NaN,NaN,ZZLN0000000079,0,Centralhub_FKHAL,Centralhub_FKHAL,SatelliteHub_KLYW,2026-11-17 15:11:53,Expected,Testorder12novT60941,2026-11-27 15:11:44,2026-11-17 15:11:53,NaN,421301,NON-FBF,Forward,Furniture,True,NaN,Vergo Bloom Dining Chair with Velvet Fabric (D...,NaN,0.0,P4,ext_mps
720583,ZZLP0000000078,NaN,NaN,NaN,NaN,ZZLN0000000079,0,Centralhub_FKHAL,Centralhub_FKHAL,SatelliteHub_KLYW,2026-11-17 15:11:54,Expected,Testorder12novT60941,2026-11-27 15:11:44,2026-11-17 15:11:54,NaN,421301,NON-FBF,Forward,Furniture,True,NaN,Vergo Bloom Dining Chair with Velvet Fabric (D...,NaN,0.0,P4,ext_mps


In [3]:
os.chdir(r"C:\Users\mohammadali1.vc\Desktop\Python data\Large Python")
# df=pd.read_csv("mps_forward_report_2026-Aug-26_12_20_01.csv")
df["merchant_code"]=df["ShipmentId"].str.slice(start=0,stop=3)
df1=df[~df['merchant_code'].isin(['FMP'])].drop_duplicates(subset='ShipmentId')
df1

,ShipmentId,CurrentHubCity,consignmentId,ConsignmentCreationTime,ConsignmentCreationHub,parentId,MPS_count,OriginHub,CurrentHubName,DestinationHub,LastUpdateTime,LatestShipmentStatus,OrderId,CPD,ShipmentCreationTime,CustomerCity,CustomerPincode,Shipment Movement,service_type,ShipmentCategory,isMPS,isPrexo,ProductDescription,OBDType,PriorityScore,PriorityScoreV2,ItemType,merchant_code
0,ALWC0000000371,NaN,NaN,NaN,NaN,ALWN0000000371,0,Centralhub_AYODHYA,Centralhub_AYODHYA,SatelliteHub_JHS,2026-10-22 09:31:15,Expected,ORD-0035822,2026-11-01 09:31:06,2026-10-22 09:31:15,NaN,284201,NON-FBF,Forward,Furniture,True,NaN,Nufield White Excel Radish Hybrid 50G,NaN,0.0,P4,ext_mps,ALW
1,ALWC0000000372,NaN,NaN,NaN,NaN,ALWN0000000371,0,Centralhub_AYODHYA,Centralhub_AYODHYA,SatelliteHub_JHS,2026-10-22 09:31:15,Expected,ORD-0035822,2026-11-01 09:31:06,2026-10-22 09:31:15,NaN,284201,NON-FBF,Forward,Furniture,True,NaN,Nufield White Excel Radish Hybrid 50G,NaN,0.0,P4,ext_mps,ALW
2,ALWC0000000373,NaN,NaN,NaN,NaN,ALWN0000000371,0,Centralhub_AYODHYA,Centralhub_AYODHYA,SatelliteHub_JHS,2026-10-22 09:31:15,Expected,ORD-0035822,2026-11-01 09:31:06,2026-10-22 09:31:15,NaN,284201,NON-FBF,Forward,Furniture,True,NaN,Nufield White Excel Radish Hybrid 50G,NaN,0.0,P4,ext_mps,ALW
3,ALWC0000000374,NaN,NaN,NaN,NaN,ALWN0000000371,0,Centralhub_AYODHYA,Centralhub_AYODHYA,SatelliteHub_JHS,2026-10-22 09:31:16,Expected,ORD-0035822,2026-11-01 09:31:06,2026-10-22 09:31:15,NaN,284201,NON-FBF,Forward,Furniture,True,NaN,Nufield White Excel Radish Hybrid 50G,NaN,0.0,P4,ext_mps,ALW
4,ALWC0000000375,NaN,NaN,NaN,NaN,ALWN0000000371,0,Centralhub_AYODHYA,Centralhub_AYODHYA,SatelliteHub_JHS,2026-10-22 09:31:16,Expected,ORD-0035822,2026-11-01 09:31:06,2026-10-22 09:31:16,NaN,284201,NON-FBF,Forward,Furniture,True,NaN,Nufield White Excel Radish Hybrid 50G,NaN,0.0,P4,ext_mps,ALW
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
720580,ZZLP0000000075,NaN,NaN,NaN,NaN,ZZLN0000000079,0,Centralhub_FKHAL,Centralhub_FKHAL,SatelliteHub_KLYW,2026-11-17 15:11:53,Expected,Testorder12novT60941,2026-11-27 15:11:44,2026-11-17 15:11:53,NaN,421301,NON-FBF,Forward,Furniture,True,NaN,Vergo Bloom Dining Chair with Velvet Fabric (D...,NaN,0.0,P4,ext_mps,ZZL
720581,ZZLP0000000076,NaN,NaN,NaN,NaN,ZZLN0000000079,0,Centralhub_FKHAL,Centralhub_FKHAL,SatelliteHub_KLYW,2026-11-17 15:11:53,Expected,Testorder12novT60941,2026-11-27 15:11:44,2026-11-17 15:11:53,NaN,421301,NON-FBF,Forward,Furniture,True,NaN,Vergo Bloom Dining Chair with Velvet Fabric (D...,NaN,0.0,P4,ext_mps,ZZL
720582,ZZLP0000000077,NaN,NaN,NaN,NaN,ZZLN0000000079,0,Centralhub_FKHAL,Centralhub_FKHAL,SatelliteHub_KLYW,2026-11-17 15:11:53,Expected,Testorder12novT60941,2026-11-27 15:11:44,2026-11-17 15:11:53,NaN,421301,NON-FBF,Forward,Furniture,True,NaN,Vergo Bloom Dining Chair with Velvet Fabric (D...,NaN,0.0,P4,ext_mps,ZZL
720583,ZZLP0000000078,NaN,NaN,NaN,NaN,ZZLN0000000079,0,Centralhub_FKHAL,Centralhub_FKHAL,SatelliteHub_KLYW,2026-11-17 15:11:54,Expected,Testorder12novT60941,2026-11-27 15:11:44,2026-11-17 15:11:54,NaN,421301,NON-FBF,Forward,Furniture,True,NaN,Vergo Bloom Dining Chair with Velvet Fabric (D...,NaN,0.0,P4,ext_mps,ZZL


In [4]:
dff=pd.read_csv("superbi_BQ_BATCH_ARKHAM_P1_SUBS_81f864e49a2a7d71a9ea09b9b2208c2a1795402265908.csv")

C:\Users\mohammadali1.vc\AppData\Local\Temp\ipykernel_22036\2523681349.py:1: DtypeWarning: Columns (0: order_id, 1: global_tracking_id, 2: product_id, 3: is_forward, 4: upstream_rto_flag, 5: global_merchant_reference_id, 6: first_rvp_pickup_time, 7: last_rvp_pickup_time, 8: actual_rvp_pickup_time, 9: rvp_completed_time) have mixed types. Specify dtype option on import or set low_memory=False.
  dff=pd.read_csv("superbi_BQ_BATCH_ARKHAM_P1_SUBS_81f864e49a2a7d71a9ea09b9b2208c2a1795402265908.csv")


In [5]:
# df3=df1.merge(dff[['tracking_id','ph_in_date_final','ph_in_time','fsd_number_of_ofd_attempts','dh_receive_time','shipped_lpd']],left_on='ShipmentId',right_on='tracking_id',how='left')
# no=pd.read_excel("Not Picked data.xlsx")
# df4=df3.merge(no,left_on='ShipmentId',right_on='AWB',how='left')
# df5=df4[~df4['Remarks'].notnull()]
# t1=df5[df5['ph_in_time'].isnull()]
# t1.to_excel("multi_track_2.xlsx")
# print("multi tracking file sent to local")

df3=df1.merge(dff[['tracking_id','ph_in_date_final','ph_in_time','fsd_number_of_ofd_attempts','dh_receive_time','shipped_lpd']],left_on='ShipmentId',right_on='tracking_id',how='left')
no=pd.read_excel("Not Picked data.xlsx")
df4=df3.merge(no,left_on='ShipmentId',right_on='AWB',how='left')
df5=df4[~df4['Remarks'].notnull()]
t1=df5[df5['ph_in_time'].isnull()]

# --- NEW CLEANUP FILTER START ---
# 1. Convert ShipmentId to string and drop rows where it is missing
t1 = t1.dropna(subset=['ShipmentId']) 
t1['ShipmentId'] = t1['ShipmentId'].astype(str).str.strip()

# 2. Keep ONLY valid Tracking IDs (must start with 4 uppercase letters followed by digits)
# This will completely drop descriptions, single numbers like "(1)", and blank rows.
valid_tid_pattern = r'^[A-Z]{4}\d+$'
t1 = t1[t1['ShipmentId'].str.match(valid_tid_pattern, na=False)]
# --- NEW CLEANUP FILTER END ---

# Save the filtered physical file
t1[['ShipmentId']].to_csv("multi_track_2.csv", index=False)

print("multi tracking file sent to local - completely cleaned!")

multi tracking file sent to local - completely cleaned!


In [6]:
import os
import shutil
import time
from math import ceil
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By 
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

# === Config ===
csv_path = r"C:\Users\mohammadali1.vc\Desktop\Python data\Large Python\multi_track_2.csv"
output_folder = (r"C:\Users\mohammadali1.vc\Desktop\Python data\Large Python\Batch Files")
# chromedriver_path = r"C:\Users\dulladeepak.vc\Desktop\Auto MT\chromedriver-win64\chromedriver.exe"
user_data_dir = (r"C:\Users\mohammadali1.vc\Desktop\Python data\Large Python\SeleniumProfile")
download_dir = r"C:\Users\mohammadali1.vc\Desktop\Python data\Large Python"
output_dir = (r"C:\Users\mohammadali1.vc\Desktop\Python data\Large Python\Tracking Output")
timeout = 10
BATCH_SIZE = 3000
MIN_FILE_SIZE_KB = 20
MAX_RETRIES = 10

# === Step 1: Read and Batch Tracking IDs ===
df = pd.read_csv(csv_path)
tracking_ids = df["ShipmentId"].dropna().astype(str).tolist()

os.makedirs(output_folder, exist_ok=True)

# 🛠️ FIX 1 & 2: Changed len(ShipmentId) to len(tracking_ids)
total_batches = ceil(len(tracking_ids) / BATCH_SIZE)
for i in range(total_batches):
    start = i * BATCH_SIZE
    end = min(start + BATCH_SIZE, len(tracking_ids))
    batch = tracking_ids[start:end]
    batch_df = pd.DataFrame(batch, columns=["ShipmentId"])
    batch_filename = f"Batch_{i+1:03}.csv"
    batch_path = os.path.join(output_folder, batch_filename)
    batch_df.to_csv(batch_path, index=False)
    print(f"✅ Saved {batch_filename} with {len(batch)} IDs")

# === Setup Chrome Driver ===
os.makedirs(output_dir, exist_ok=True)
options = Options()
options.add_argument("--start-maximized")
options.add_argument(f"--user-data-dir={user_data_dir}")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)
prefs = {
    "download.default_directory": download_dir,
    "download.prompt_for_download": False,
    "download.directory_upgrade": True,
}
options.add_experimental_option("prefs", prefs)

# driver = webdriver.Chrome(service=Service(chromedriver_path), options=options)
driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, timeout)

# === Open and Login ===
driver.get("http://10.24.1.71/hub-ops/tracking/shipments")
time.sleep(3)
try:
    username_input = driver.find_element(By.ID, "username")
    username_input.send_keys("mohammadali1.vc@flipkart.com")
    driver.find_element(By.ID, "password").send_keys("KKptnRF9FWaN#2026")
    driver.find_element(By.ID, "loginSubmit").click()
    input("🔐 Complete 2FA manually, then press Enter to continue...")
except:
    print("✅ Already logged in or login not required.")

# === Navigation ===
try:
    wait.until(
        EC.element_to_be_clickable(
            (By.XPATH, "//div[contains(text(),'Single Shipment Tracking')]")
        )
    ).click()
    print("✅ Clicked 'Single Shipment Tracking'")
    wait.until(
        EC.element_to_be_clickable(
            (By.XPATH, "//div[contains(text(),'Multiple Shipment Tracking')]")
        )
    ).click()
    print("✅ Clicked 'Multiple Shipment Tracking'")
except Exception as e:
    print("❌ Navigation failed:", e)
    driver.quit()
    raise


# === Helpers ===
def get_input_field():
    container = wait.until(
        EC.presence_of_element_located(
            (
                By.CSS_SELECTOR,
                "div[data-testid='selectComponent-shipmentsToBeTracked']",
            )
        )
    )
    return container.find_element(
        By.CSS_SELECTOR, "input[id^='react-select-'][type='text']"
    )


def clear_input_field():
    try:
        clear_button = wait.until(
            EC.element_to_be_clickable(
                (
                    By.XPATH,
                    "//button[contains(@class, 'jNFxRN') and contains(text(), 'Clear')]",
                )
            )
        )
        clear_button.click()
        time.sleep(1)
        print("🧹 Cleared input field using Clear button")
    except:
        input_field = get_input_field()
        input_field.send_keys(Keys.CONTROL + "a")
        input_field.send_keys(Keys.DELETE)
        print("🧹 Fallback: Cleared input field using keyboard")
        time.sleep(0.5)


def insert_ids_in_batch(ids):
    try:
        input_field = get_input_field()
        ids_string = ",".join(ids)
        driver.execute_script("arguments[0].focus();", input_field)
        driver.execute_script("arguments[0].value = '';", input_field)
        driver.execute_script(
            """
            const input = arguments[0];
            const lastValue = input.value;
            input.value = arguments[1];
            const event = new Event('input', { bubbles: true });
            event.simulated = true;
            const tracker = input._valueTracker;
            if (tracker) {
                tracker.setValue(lastValue);
            }
            input.dispatchEvent(event);
        """,
            input_field,
            ids_string,
        )
        entered_value = driver.execute_script(
            "return arguments[0].value", input_field
        )
        print(
            f"⚠️ Inserted {len(ids)} IDs (actual input count: {len(entered_value.split(','))})"
        )
        return True
    except Exception as e:
        print(f"❌ Error during insertion: {str(e)}")
        return False


def click_track_button():
    try:
        wait.until(
            EC.element_to_be_clickable(
                (By.XPATH, "//button[contains(text(),'Track')]")
            )
        ).click()
        print("📥 Clicked 'Track'")
        wait.until(
            EC.element_to_be_clickable(
                (By.XPATH, "//button[contains(text(),'Download')]")
            )
        )
        return True
    except Exception as e:
        print(f"❌ Track click failed: {e}")
        return False


def download_and_verify():
    try:
        before = set(os.listdir(download_dir))
        wait.until(
            EC.element_to_be_clickable(
                (By.XPATH, "//button[contains(text(),'Download')]")
            )
        ).click()
        print("📁 Clicked 'Download'")
        for _ in range(30):
            after = set(os.listdir(download_dir))
            new_files = list(after - before)
            new_files = [f for f in new_files if not f.endswith(".crdownload")]
            if new_files:
                file_path = os.path.join(download_dir, new_files[0])
                time.sleep(3)
                if (
                    os.path.exists(file_path)
                    and os.path.getsize(file_path) > MIN_FILE_SIZE_KB * 1024
                ):
                    new_name = f"batch_{time.strftime('%Y%m%d_%H%M%S')}_{new_files[0]}"
                    shutil.move(file_path, os.path.join(output_dir, new_name))
                    print(f"✅ Downloaded and saved as {new_name}")
                    return True
                elif os.path.exists(file_path):
                    os.remove(file_path)
                    print("⚠️ Small or corrupt file deleted")
                    return False
            time.sleep(1)
        print("❌ Download timeout")
        return False
    except Exception as e:
        print(f"❌ Download error: {e}")
        return False


# === Process All Batches With Retry ===
batch_files = sorted(
    [f for f in os.listdir(output_folder) if f.endswith(".csv")]
)
for file in batch_files:
    print(f"\n🚚 Processing {file}")
    df_batch = pd.read_csv(os.path.join(output_folder, file))

    # 🛠️ FIX 3: Changed 'tracking_id' to 'ShipmentId' to match the column name generated above
    ids = df_batch["ShipmentId"].dropna().astype(str).tolist()

    for attempt in range(1, MAX_RETRIES + 1):
        print(f"🔁 Attempt {attempt} for {file}")
        clear_input_field()
        if insert_ids_in_batch(ids) and click_track_button():
            if download_and_verify():
                break  # Success!
        if attempt == MAX_RETRIES:
            print(
                f"❌ Failed to download {file} after {MAX_RETRIES} attempts. Skipping..."
            )

driver.quit()
print("\n🎉 ALL DONE")

✅ Saved Batch_001.csv with 3000 IDs
✅ Saved Batch_002.csv with 3000 IDs
✅ Saved Batch_003.csv with 3000 IDs
✅ Saved Batch_004.csv with 3000 IDs
✅ Saved Batch_005.csv with 3000 IDs
✅ Saved Batch_006.csv with 3000 IDs
✅ Saved Batch_007.csv with 3000 IDs
✅ Saved Batch_008.csv with 3000 IDs
✅ Saved Batch_009.csv with 3000 IDs
✅ Saved Batch_010.csv with 3000 IDs
✅ Saved Batch_011.csv with 3000 IDs
✅ Saved Batch_012.csv with 1184 IDs


🔐 Complete 2FA manually, then press Enter to continue... 


✅ Clicked 'Single Shipment Tracking'
✅ Clicked 'Multiple Shipment Tracking'

🚚 Processing Batch_001.csv
🔁 Attempt 1 for Batch_001.csv
🧹 Cleared input field using Clear button
⚠️ Inserted 3000 IDs (actual input count: 1)
📥 Clicked 'Track'
📁 Clicked 'Download'
✅ Downloaded and saved as batch_20261123_093136_shipments 11_23_2026, 9_31_32 AM.csv

🚚 Processing Batch_002.csv
🔁 Attempt 1 for Batch_002.csv
🧹 Cleared input field using Clear button
⚠️ Inserted 3000 IDs (actual input count: 1)
📥 Clicked 'Track'
📁 Clicked 'Download'
❌ Download timeout
🔁 Attempt 2 for Batch_002.csv
🧹 Cleared input field using Clear button
⚠️ Inserted 3000 IDs (actual input count: 1)
📥 Clicked 'Track'
📁 Clicked 'Download'
✅ Downloaded and saved as batch_20261123_093227_shipments 11_23_2026, 9_32_23 AM.csv

🚚 Processing Batch_003.csv
🔁 Attempt 1 for Batch_003.csv
🧹 Cleared input field using Clear button
⚠️ Inserted 3000 IDs (actual input count: 1)
📥 Clicked 'Track'
📁 Clicked 'Download'
✅ Downloaded and saved as batch

In [7]:
folder_path = r"C:\Users\mohammadali1.vc\Desktop\Python data\Large Python\Tracking Output"
import glob  # Add this line first

csv_files = glob.glob(os.path.join(folder_path, '*.csv'))

dfs = []

for file_path in csv_files:
    df = pd.read_csv(file_path)
    dfs.append(df)

merged_data = pd.concat(dfs, ignore_index=True)

In [8]:
os.chdir(r"C:\Users\mohammadali1.vc\Desktop\Python data\Large Python")

In [9]:
import numpy as np

In [10]:
merged_data['remark_1']=np.where((merged_data['First Receive Time'].isnull()),'Not_picked',None)
df6=df4.merge(merged_data[['Tracking Id','Latest Status','Delivery Hub','Current Location','Latest Update Time','First Receive Time','Logistics Promise Date','No. of Attempts','remark_1']],left_on='ShipmentId',right_on='Tracking Id',how='left').drop_duplicates(subset='ShipmentId')
df6['ph_in_time']=df6['ph_in_time'].fillna(df6['First Receive Time'])
#df6['LastUpdateTime'] = pd.to_datetime(df6['LastUpdateTime'],format='%Y-%m-%d').dt.date
df6['LastUpdateTime'] = pd.to_datetime(df6['LastUpdateTime'].str.split().str[0], format='%Y-%m-%d').dt.date
df6['todays_date']=datetime.today().strftime('%Y-%m-%d')
#df6['todays_date']='2024-04-29'
ageing_days=(pd.to_datetime(df6['todays_date']) - pd.to_datetime(df6['LastUpdateTime'])).dt.days
df6["Last_Scan_ageing"]=ageing_days
df6['CurrentHubName']=df6['CurrentHubName'].str.upper()
#df8=df6[~df6['remark_1'].isin([9])]
df8=df6.copy()
df8['OriginHub']=df8['OriginHub'].str.upper()
df8['CurrentHubName']=df8['CurrentHubName'].str.upper()
df8['Hub_check']=df8['OriginHub']==df8['CurrentHubName']


lg=pd.read_excel("Large XD mapping.xlsx")
lg['Hub name']=lg['Hub name'].str.upper()
df7=df8.merge(lg[['Hub name']],left_on='CurrentHubName',right_on='Hub name',how='left')
# conditions = [
#     # 1. Move AT FM rules to the very top
#     (df7['Hub_check'] == True) & (df7['LatestShipmentStatus'] == 'Undelivered_Not_Attended'),
#     (df7['Hub_check'] == True) & (df7['LatestShipmentStatus'] == 'Expected') & (df7['ph_in_time'].notnull()),
    
#     (df7['Hub_check'] == True) & (df7['LatestShipmentStatus'] == 'Expected') & (df7['ph_in_time'].isnull()) & (df7['ph_in_date_final'].isnull()) & (df7['dh_receive_time'].isnull()),
#     (df7['LatestShipmentStatus'] == 'Undelivered_Order_Rejected_By_Customer'),
#     (df7['fsd_number_of_ofd_attempts'] == 1),
#     (df7['fsd_number_of_ofd_attempts'] == 2),
#     (df7['fsd_number_of_ofd_attempts'] > 2),
#     (df7['fsd_number_of_ofd_attempts'] == 0) & (df7['LatestShipmentStatus'] == 'Undelivered_NonServiceablePincode'),
#     (df7['LatestShipmentStatus'] == 'Undelivered_SameStateMisroute'),
#     (df7['LatestShipmentStatus'] == 'Undelivered_OtherStateMisroute'),
#     ((df7['CurrentHubName'].str.startswith(('SATELLITEHUB', 'BULKHUB'))) & (~df7['LatestShipmentStatus'].isin(['Expected'])) & (df7['Last_Scan_ageing'] <= 2)),
#     ((df7['CurrentHubName'].str.startswith(('SATELLITEHUB', 'BULKHUB'))) & (~df7['LatestShipmentStatus'].isin(['Expected'])) & (df7['Last_Scan_ageing'] > 2)),
#     ((df7['CurrentHubName'].str.startswith(('SATELLITEHUB', 'BULKHUB', 'SATELLITEHUB_'))) & (df7['LatestShipmentStatus'].isin(['Expected']))),
#     ((df7['CurrentHubName'].str.startswith(('CENTRAL'))) & (df7['LatestShipmentStatus'].isin(['Expected']))),
#     (df7['Hub name'].notnull()) & (df7['LatestShipmentStatus'].isin(['Expected'])),
#     (df7['Hub name'].notnull()) & (~df7['LatestShipmentStatus'].isin(['Expected'])),
#     ((df7['CurrentHubName'].str.startswith(('CENTRAL'))) & (~df7['LatestShipmentStatus'].isin(['Expected']))),                                                                                                     
#     (df7['Hub_check'] == True ) & df7['LatestShipmentStatus'].isin(['Undelivered_Not_Attended']),
#     (df7['Hub_check'] == True) & (df7['LatestShipmentStatus'] == 'Expected') & (df7['ph_in_time'].notnull())
# ]

# choices = [
#     # Match the choices to the top positions
#     'AT FM',
#     'AT FM',
    
#     # The rest of your choices remain in the exact same relative order
#     'Not_picked',
#     'ORC',
#     'NDR-1',
#     'NDR-2',
#     '>3attempts',
#     'NSS',
#     'SSM',
#     'OSM',
#     'Freshlanding',
#     'NCD>48hrs',
#     'Intransit to DH',
#     'Intransit to XD',
#     'Intransit to XD',
#     'At XD',
#     'At XD'
# ]

# df7['Remark'] = np.select(conditions, choices, default=None)

# df7['remark_1'] = df7['Remark'].fillna(df7['remark_1'])
# # df7['remark_1']=df7['remark_1'].fillna(df7['Remark'])
# print(df7.remark_1.value_counts())

# Make sure BOTH lists have exactly 17 elements
conditions = [
    # 1 & 2: AT FM Rules (Priority)
    (df7['Hub_check'] == True) & (df7['LatestShipmentStatus'] == 'Undelivered_Not_Attended'),
    (df7['Hub_check'] == True) & (df7['LatestShipmentStatus'] == 'Expected') & (df7['ph_in_time'].notnull()),
    
    # 3 to 10: Exception & Delivery Rules
    (df7['Hub_check'] == True) & (df7['LatestShipmentStatus'] == 'Expected') & (df7['ph_in_time'].isnull()) & (df7['ph_in_date_final'].isnull()) & (df7['dh_receive_time'].isnull()),
    (df7['LatestShipmentStatus'] == 'Undelivered_Order_Rejected_By_Customer'),
    (df7['fsd_number_of_ofd_attempts'] == 1),
    (df7['fsd_number_of_ofd_attempts'] == 2),
    (df7['fsd_number_of_ofd_attempts'] > 2),
    (df7['fsd_number_of_ofd_attempts'] == 0) & (df7['LatestShipmentStatus'] == 'Undelivered_NonServiceablePincode'),
    (df7['LatestShipmentStatus'] == 'Undelivered_SameStateMisroute'),
    (df7['LatestShipmentStatus'] == 'Undelivered_OtherStateMisroute'),
    
    # 11 to 13: Satellite & Bulk Hub Rules
    ((df7['CurrentHubName'].str.startswith(('SATELLITEHUB', 'BULKHUB'))) & (~df7['LatestShipmentStatus'].isin(['Expected'])) & (df7['Last_Scan_ageing'] <= 2)),
    ((df7['CurrentHubName'].str.startswith(('SATELLITEHUB', 'BULKHUB'))) & (~df7['LatestShipmentStatus'].isin(['Expected'])) & (df7['Last_Scan_ageing'] > 2)),
    ((df7['CurrentHubName'].str.startswith(('SATELLITEHUB', 'BULKHUB', 'SATELLITEHUB_'))) & (df7['LatestShipmentStatus'].isin(['Expected']))),
    
    # 14 to 17: Central & Cross-Dock (XD) Hub Rules
    ((df7['CurrentHubName'].str.startswith(('CENTRAL'))) & (df7['LatestShipmentStatus'].isin(['Expected']))),
    (df7['Hub name'].notnull()) & (df7['LatestShipmentStatus'].isin(['Expected'])),
    (df7['Hub name'].notnull()) & (~df7['LatestShipmentStatus'].isin(['Expected'])),
    ((df7['CurrentHubName'].str.startswith(('CENTRAL'))) & (~df7['LatestShipmentStatus'].isin(['Expected'])))
]

choices = [
    # 1 & 2
    'AT FM', 
    'AT FM',
    
    # 3 to 10
    'Not_picked', 
    'ORC', 
    'NDR-1', 
    'NDR-2', 
    '>3attempts', 
    'NSS', 
    'SSM', 
    'OSM',
    
    # 11 to 13
    'Freshlanding', 
    'NCD>48hrs', 
    'Intransit to DH',
    
    # 14 to 17
    'Intransit to XD', 
    'Intransit to XD', 
    'At XD', 
    'At XD'
]

# This will run smoothly now
df7['Remark'] = np.select(conditions, choices, default=None)

# Prioritize your new rules over the old merge column
df7['remark_1'] = df7['Remark'].fillna(df7['remark_1'])

print(df7['remark_1'].value_counts())

df7['shipped_lpd']=pd.to_datetime(df7['shipped_lpd']).dt.date
ageing=(pd.to_datetime(df7['shipped_lpd']) - pd.to_datetime(df7['todays_date'])).dt.days
df7["chek_col"]=ageing
df7["chek_col"] = np.where(df7["chek_col"].isna(),(pd.to_datetime(df7['shipped_lpd']) - pd.to_datetime(df7['todays_date'])).dt.days,df7["chek_col"])
df7['LPD']=np.where(df7['chek_col'] < 0,'Past LPD',
                   np.where(df7['chek_col'] == 0 ,'Todays LPD',
                           np.where(df7['chek_col'] >0,'Future LPD',None)))

remark_1
Intransit to XD    22771
Not_picked         21586
Freshlanding       17145
At XD              13836
Intransit to DH     2609
NCD>48hrs           2087
NDR-1               1769
AT FM                947
>3attempts           855
NDR-2                523
ORC                   46
OSM                   40
NSS                    8
SSM                    1
Name: count, dtype: int64


In [11]:
df7

,ShipmentId,CurrentHubCity,consignmentId,ConsignmentCreationTime,ConsignmentCreationHub,parentId,MPS_count,OriginHub,CurrentHubName,DestinationHub,LastUpdateTime,LatestShipmentStatus,OrderId,CPD,ShipmentCreationTime,CustomerCity,CustomerPincode,Shipment Movement,service_type,ShipmentCategory,isMPS,isPrexo,ProductDescription,OBDType,PriorityScore,PriorityScoreV2,ItemType,merchant_code,tracking_id,ph_in_date_final,ph_in_time,fsd_number_of_ofd_attempts,dh_receive_time,shipped_lpd,AWB,Remarks,Tracking Id,Latest Status,Delivery Hub,Current Location,Latest Update Time,First Receive Time,Logistics Promise Date,No. of Attempts,remark_1,todays_date,Last_Scan_ageing,Hub_check,Hub name,Remark,chek_col,LPD
0,ALWC0000000371,NaN,NaN,NaN,NaN,ALWN0000000371,0,CENTRALHUB_AYODHYA,CENTRALHUB_AYODHYA,SatelliteHub_JHS,2026-10-22,Expected,ORD-0035822,2026-11-01 09:31:06,2026-10-22 09:31:15,NaN,284201,NON-FBF,Forward,Furniture,True,NaN,Nufield White Excel Radish Hybrid 50G,NaN,0.0,P4,ext_mps,ALW,ALWC0000000371,NaN,NaN,0.0,NaN,NaT,NaN,NaN,ALWC0000000371,Expected,SatelliteHub_JHS,Centralhub_AYODHYA,2026-10-22 09:31:15,NaN,NaN,0.0,Not_picked,2026-11-23,32,True,NaN,Not_picked,NaN,NaN
1,ALWC0000000372,NaN,NaN,NaN,NaN,ALWN0000000371,0,CENTRALHUB_AYODHYA,CENTRALHUB_AYODHYA,SatelliteHub_JHS,2026-10-22,Expected,ORD-0035822,2026-11-01 09:31:06,2026-10-22 09:31:15,NaN,284201,NON-FBF,Forward,Furniture,True,NaN,Nufield White Excel Radish Hybrid 50G,NaN,0.0,P4,ext_mps,ALW,ALWC0000000372,NaN,NaN,0.0,NaN,NaT,NaN,NaN,ALWC0000000372,Expected,SatelliteHub_JHS,Centralhub_AYODHYA,2026-10-22 09:31:15,NaN,NaN,0.0,Not_picked,2026-11-23,32,True,NaN,Not_picked,NaN,NaN
2,ALWC0000000373,NaN,NaN,NaN,NaN,ALWN0000000371,0,CENTRALHUB_AYODHYA,CENTRALHUB_AYODHYA,SatelliteHub_JHS,2026-10-22,Expected,ORD-0035822,2026-11-01 09:31:06,2026-10-22 09:31:15,NaN,284201,NON-FBF,Forward,Furniture,True,NaN,Nufield White Excel Radish Hybrid 50G,NaN,0.0,P4,ext_mps,ALW,ALWC0000000373,NaN,NaN,0.0,NaN,NaT,NaN,NaN,ALWC0000000373,Expected,SatelliteHub_JHS,Centralhub_AYODHYA,2026-10-22 09:31:15,NaN,NaN,0.0,Not_picked,2026-11-23,32,True,NaN,Not_picked,NaN,NaN
3,ALWC0000000374,NaN,NaN,NaN,NaN,ALWN0000000371,0,CENTRALHUB_AYODHYA,CENTRALHUB_AYODHYA,SatelliteHub_JHS,2026-10-22,Expected,ORD-0035822,2026-11-01 09:31:06,2026-10-22 09:31:15,NaN,284201,NON-FBF,Forward,Furniture,True,NaN,Nufield White Excel Radish Hybrid 50G,NaN,0.0,P4,ext_mps,ALW,ALWC0000000374,NaN,NaN,0.0,NaN,NaT,NaN,NaN,ALWC0000000374,Expected,SatelliteHub_JHS,Centralhub_AYODHYA,2026-10-22 09:31:16,NaN,NaN,0.0,Not_picked,2026-11-23,32,True,NaN,Not_picked,NaN,NaN
4,ALWC0000000375,NaN,NaN,NaN,NaN,ALWN0000000371,0,CENTRALHUB_AYODHYA,CENTRALHUB_AYODHYA,SatelliteHub_JHS,2026-10-22,Expected,ORD-0035822,2026-11-01 09:31:06,2026-10-22 09:31:16,NaN,284201,NON-FBF,Forward,Furniture,True,NaN,Nufield White Excel Radish Hybrid 50G,NaN,0.0,P4,ext_mps,ALW,ALWC0000000375,NaN,NaN,0.0,NaN,NaT,NaN,NaN,ALWC0000000375,Expected,SatelliteHub_JHS,Centralhub_AYODHYA,2026-10-22 09:31:16,NaN,NaN,0.0,Not_picked,2026-11-23,32,True,NaN,Not_picked,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84236,ZZLP0000000075,NaN,NaN,NaN,NaN,ZZLN0000000079,0,CENTRALHUB_FKHAL,CENTRALHUB_FKHAL,SatelliteHub_KLYW,2026-11-17,Expected,Testorder12novT60941,2026-11-27 15:11:44,2026-11-17 15:11:53,NaN,421301,NON-FBF,Forward,Furniture,True,NaN,Vergo Bloom Dining Chair with Velvet Fabric (D...,NaN,0.0,P4,ext_mps,ZZL,ZZLP0000000075,NaN,NaN,0.0,NaN,NaT,NaN,NaN,ZZLP0000000075,Expected,SatelliteHub_KLYW,Centralhub_FKHAL,2026-11-17 15:11:53,NaN,NaN,0.0,Not_picked,2026-11-23,6,True,NaN,Not_picked,NaN,NaN
84237,ZZLP0000000076,NaN,NaN,NaN,NaN,ZZLN0000000079,0,CENTRALHUB_FKHAL,CENTRALHUB_FKHAL,SatelliteHub_KLYW,2026-11-17,Expected,Testorder12novT60941,2026-11-27 15:11:44,2026-11-17 15:11:53,NaN,421301,NON-FBF,Forward,Furniture,True,NaN

In [12]:
# df7.to_csv("Pincode_updating.csv",index=False)

In [13]:
# df7= pd.read_csv("Pincode_updating.csv")
# df7AssertionError

In [14]:
# Keep only rows where remark_1 is NOT 'Not_picked'
df7 = df7[df7['remark_1'] != 'Not_picked']

In [15]:
lar=pd.read_excel("Large_mapping_2.xlsx")
lar['Pincode']=lar['Pincode'].astype(str)
df7['CustomerPincode']=df7['CustomerPincode'].astype(str).str.strip().str.lstrip("'")
df7['CustomerPincode'] = df7['CustomerPincode'].str.extract(r'(\d+)', expand=False).astype(str)
df10=df7.merge(lar,left_on='CustomerPincode',right_on='Pincode',how='left')

In [16]:

df10['check']=np.where((df10['remark_1'].isin(['Freshlanding','NCD>48hrs','NDR-1','NDR-2','NSS','OSM','SSM'])),'Blank',None)
df10.check.value_counts()

check
Blank    21575
Name: count, dtype: int64

In [17]:
# 1. Sort the dataframe by parentId column from A to Z
df10 = df10.sort_values(by='parentId', ascending=True)
df10

,ShipmentId,CurrentHubCity,consignmentId,ConsignmentCreationTime,ConsignmentCreationHub,parentId,MPS_count,OriginHub,CurrentHubName,DestinationHub,LastUpdateTime,LatestShipmentStatus,OrderId,CPD,ShipmentCreationTime,CustomerCity,CustomerPincode,Shipment Movement,service_type,ShipmentCategory,isMPS,isPrexo,ProductDescription,OBDType,PriorityScore,PriorityScoreV2,ItemType,merchant_code,tracking_id,ph_in_date_final,ph_in_time,fsd_number_of_ofd_attempts,dh_receive_time,shipped_lpd,AWB,Remarks,Tracking Id,Latest Status,Delivery Hub,Current Location,Latest Update Time,First Receive Time,Logistics Promise Date,No. of Attempts,remark_1,todays_date,Last_Scan_ageing,Hub_check,Hub name,Remark,chek_col,LPD,Pincode,Zone,State,Beat,Hub Name,GM,check
77,ALWC0000027756,NaN,NaN,NaN,NaN,ALWC0000027756,0,CENTRALHUB_BLPFM,CENTRALHUB_L_DEL2,SATELLITEHUB_FKKRKL,2026-11-23,Undelivered_Not_Attended,ORD-0075987,2026-12-04 23:59:59,2026-11-22 12:21:17,NaN,614809,NON-FBF,Forward,Furniture,True,NaN,NACL Profex Super 1LTR,NaN,0.0,P4,ext_sps,ALW,ALWC0000027756,2026-11-22,2026-11-22T18:01:21.000+05:30,0.0,NaN,2026-12-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,At XD,2026-11-23,0,False,NaN,At XD,11.0,Future LPD,614809,South,TAMIL NADU,VLG,SATELLITEHUB_FKKRKL,Ajay G Menon,NaN
78,ALWC0000027759,NaN,NaN,NaN,NaN,ALWC0000027759,0,CENTRALHUB_BLPFM,CENTRALHUB_L_BLR5,Satellitehub_RCTY,2026-11-23,Expected,ORD-0075775,2026-12-03 23:59:59,2026-11-22 11:27:47,NaN,517236,NON-FBF,Forward,Furniture,True,NaN,NACL Profex Super 500ML,NaN,0.0,P4,ext_sps,ALW,ALWC0000027759,2026-11-22,2026-11-22T18:01:21.000+05:30,0.0,NaN,2026-12-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Intransit to XD,2026-11-23,0,False,NaN,Intransit to XD,10.0,Future LPD,517236,South,ANDHRA PRADESH,TMP,SATELLITEHUB_RCTY,Deepak Kumar,NaN
79,ALWC0000027760,NaN,NaN,NaN,NaN,ALWC0000027760,0,CENTRALHUB_BLPFM,CENTRALHUB_L_DEL2,Satellitehub_BDPR1,2026-11-23,Undelivered_Not_Attended,ORD-0075384,2026-12-01 23:59:59,2026-11-22 10:52:28,NaN,788806,NON-FBF,Forward,Furniture,True,NaN,Hymatic Charger 2 Amp Sprayer Spare Parts,NaN,0.0,P4,ext_sps,ALW,ALWC0000027760,2026-11-22,2026-11-22T18:01:25.000+05:30,0.0,NaN,2026-12-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,At XD,2026-11-23,0,False,NaN,At XD,8.0,Future LPD,788806,East,Assam,BDP,Satellitehub_BDPR1,Suranjit Chatterjee,NaN
80,ALWC0000027762,NaN,NaN,NaN,NaN,ALWC0000027762,0,CENTRALHUB_KNL1,CENTRALHUB_L_DEL2,SatelliteHub_VNS,2026-11-23,Expected,ORD-0076113,2026-12-02 23:59:59,2026-11-21 23:42:14,NaN,231306,NON-FBF,Forward,Furniture,True,NaN,Shriram Fiprid 100G,NaN,0.0,P4,ext_sps,ALW,ALWC0000027762,2026-11-22,2026-11-22T11:42:49.000+05:30,0.0,NaN,2026-12-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Intransit to XD,2026-11-23,0,False,NaN,Intransit to XD,9.0,Future LPD,231306,North,UTTAR PRADESH,RJA,SATELLITEHUB_VNS,DILIP KUMAR,NaN
81,ALWC0000027764,NaN,NaN,NaN,NaN,ALWC0000027764,0,CENTRALHUB_SY_RJK,CENTRALHUB_L_AMD3,Satellitehub_FKGJB,2026-11-23,Expected,ORD-0075827,2026-12-06 23:59:59,2026-11-21 19:41:00,NaN,464331,NON-FBF,Forward,Furniture,True,NaN,Star Agri Solutions Star Claim 500G,NaN,0.0,P4,ext_sps,ALW,ALWC0000027764,2026-11-22,2026-11-22T15:32:39.000+05:30,0.0,NaN,2026-12-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Intransit to XD,2026-11-23,0,False,CENTRALHUB_L_AMD3,Intransit to XD,13.0,Future LPD,464331,West,Madhya Pradesh,GYA,Satellitehub_FKGJB,Kishore Asrani,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62652,ZHBP0000000061,NaN,NaN,NaN,NaN,ZHBP0000000061,0,CENTRALHUB_STV1,CENTRALHUB_L_MUMX,SatelliteHub_GUW,2026-11-23,Undelivered_Not_Attended,639444461130,2026-11-30 23:59:59,2026-11-20 14:11:37,NaN,781035,NON-FBF,Forward,Furniture,True,NaN,Medical Equipment,NaN,0.0,P4,ext_sps,ZHB,ZHBP0000000061,2026-11-21,2026-11-21T15:53:12.000+05:30,0.0,NaN,2026-11-30,NaN,NaN,NaN,NaN,NaN,N

In [18]:
# 1. Clean up parentId column to keep only valid structural IDs
df10 = df10.dropna(subset=['parentId'])
df10['parentId'] = df10['parentId'].astype(str).str.strip()

# Filters out row anomalies like single digits "1" or text descriptions
# Keeps only true numerical IDs or standard corporate alphanumeric sequences
df10 = df10[df10['parentId'].str.match(r'^(?!\d$)[A-Za-z0-9_-]+$', na=False)]

# 2. Sort from A to Z
df10 = df10.sort_values(by='parentId', ascending=True)

In [19]:
df10

,ShipmentId,CurrentHubCity,consignmentId,ConsignmentCreationTime,ConsignmentCreationHub,parentId,MPS_count,OriginHub,CurrentHubName,DestinationHub,LastUpdateTime,LatestShipmentStatus,OrderId,CPD,ShipmentCreationTime,CustomerCity,CustomerPincode,Shipment Movement,service_type,ShipmentCategory,isMPS,isPrexo,ProductDescription,OBDType,PriorityScore,PriorityScoreV2,ItemType,merchant_code,tracking_id,ph_in_date_final,ph_in_time,fsd_number_of_ofd_attempts,dh_receive_time,shipped_lpd,AWB,Remarks,Tracking Id,Latest Status,Delivery Hub,Current Location,Latest Update Time,First Receive Time,Logistics Promise Date,No. of Attempts,remark_1,todays_date,Last_Scan_ageing,Hub_check,Hub name,Remark,chek_col,LPD,Pincode,Zone,State,Beat,Hub Name,GM,check
77,ALWC0000027756,NaN,NaN,NaN,NaN,ALWC0000027756,0,CENTRALHUB_BLPFM,CENTRALHUB_L_DEL2,SATELLITEHUB_FKKRKL,2026-11-23,Undelivered_Not_Attended,ORD-0075987,2026-12-04 23:59:59,2026-11-22 12:21:17,NaN,614809,NON-FBF,Forward,Furniture,True,NaN,NACL Profex Super 1LTR,NaN,0.0,P4,ext_sps,ALW,ALWC0000027756,2026-11-22,2026-11-22T18:01:21.000+05:30,0.0,NaN,2026-12-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,At XD,2026-11-23,0,False,NaN,At XD,11.0,Future LPD,614809,South,TAMIL NADU,VLG,SATELLITEHUB_FKKRKL,Ajay G Menon,NaN
78,ALWC0000027759,NaN,NaN,NaN,NaN,ALWC0000027759,0,CENTRALHUB_BLPFM,CENTRALHUB_L_BLR5,Satellitehub_RCTY,2026-11-23,Expected,ORD-0075775,2026-12-03 23:59:59,2026-11-22 11:27:47,NaN,517236,NON-FBF,Forward,Furniture,True,NaN,NACL Profex Super 500ML,NaN,0.0,P4,ext_sps,ALW,ALWC0000027759,2026-11-22,2026-11-22T18:01:21.000+05:30,0.0,NaN,2026-12-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Intransit to XD,2026-11-23,0,False,NaN,Intransit to XD,10.0,Future LPD,517236,South,ANDHRA PRADESH,TMP,SATELLITEHUB_RCTY,Deepak Kumar,NaN
79,ALWC0000027760,NaN,NaN,NaN,NaN,ALWC0000027760,0,CENTRALHUB_BLPFM,CENTRALHUB_L_DEL2,Satellitehub_BDPR1,2026-11-23,Undelivered_Not_Attended,ORD-0075384,2026-12-01 23:59:59,2026-11-22 10:52:28,NaN,788806,NON-FBF,Forward,Furniture,True,NaN,Hymatic Charger 2 Amp Sprayer Spare Parts,NaN,0.0,P4,ext_sps,ALW,ALWC0000027760,2026-11-22,2026-11-22T18:01:25.000+05:30,0.0,NaN,2026-12-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,At XD,2026-11-23,0,False,NaN,At XD,8.0,Future LPD,788806,East,Assam,BDP,Satellitehub_BDPR1,Suranjit Chatterjee,NaN
80,ALWC0000027762,NaN,NaN,NaN,NaN,ALWC0000027762,0,CENTRALHUB_KNL1,CENTRALHUB_L_DEL2,SatelliteHub_VNS,2026-11-23,Expected,ORD-0076113,2026-12-02 23:59:59,2026-11-21 23:42:14,NaN,231306,NON-FBF,Forward,Furniture,True,NaN,Shriram Fiprid 100G,NaN,0.0,P4,ext_sps,ALW,ALWC0000027762,2026-11-22,2026-11-22T11:42:49.000+05:30,0.0,NaN,2026-12-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Intransit to XD,2026-11-23,0,False,NaN,Intransit to XD,9.0,Future LPD,231306,North,UTTAR PRADESH,RJA,SATELLITEHUB_VNS,DILIP KUMAR,NaN
81,ALWC0000027764,NaN,NaN,NaN,NaN,ALWC0000027764,0,CENTRALHUB_SY_RJK,CENTRALHUB_L_AMD3,Satellitehub_FKGJB,2026-11-23,Expected,ORD-0075827,2026-12-06 23:59:59,2026-11-21 19:41:00,NaN,464331,NON-FBF,Forward,Furniture,True,NaN,Star Agri Solutions Star Claim 500G,NaN,0.0,P4,ext_sps,ALW,ALWC0000027764,2026-11-22,2026-11-22T15:32:39.000+05:30,0.0,NaN,2026-12-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Intransit to XD,2026-11-23,0,False,CENTRALHUB_L_AMD3,Intransit to XD,13.0,Future LPD,464331,West,Madhya Pradesh,GYA,Satellitehub_FKGJB,Kishore Asrani,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62652,ZHBP0000000061,NaN,NaN,NaN,NaN,ZHBP0000000061,0,CENTRALHUB_STV1,CENTRALHUB_L_MUMX,SatelliteHub_GUW,2026-11-23,Undelivered_Not_Attended,639444461130,2026-11-30 23:59:59,2026-11-20 14:11:37,NaN,781035,NON-FBF,Forward,Furniture,True,NaN,Medical Equipment,NaN,0.0,P4,ext_sps,ZHB,ZHBP0000000061,2026-11-21,2026-11-21T15:53:12.000+05:30,0.0,NaN,2026-11-30,NaN,NaN,NaN,NaN,NaN,N

In [20]:
# Drop the Pincode and Hubname column completely from the dataframe
df10 = df10.drop(columns=['Pincode'], errors='ignore')
# Drop the first HUBname column specifically
df10 = df10.drop(columns=['Hub name'], errors='ignore')

In [21]:
df10

,ShipmentId,CurrentHubCity,consignmentId,ConsignmentCreationTime,ConsignmentCreationHub,parentId,MPS_count,OriginHub,CurrentHubName,DestinationHub,LastUpdateTime,LatestShipmentStatus,OrderId,CPD,ShipmentCreationTime,CustomerCity,CustomerPincode,Shipment Movement,service_type,ShipmentCategory,isMPS,isPrexo,ProductDescription,OBDType,PriorityScore,PriorityScoreV2,ItemType,merchant_code,tracking_id,ph_in_date_final,ph_in_time,fsd_number_of_ofd_attempts,dh_receive_time,shipped_lpd,AWB,Remarks,Tracking Id,Latest Status,Delivery Hub,Current Location,Latest Update Time,First Receive Time,Logistics Promise Date,No. of Attempts,remark_1,todays_date,Last_Scan_ageing,Hub_check,Remark,chek_col,LPD,Zone,State,Beat,Hub Name,GM,check
77,ALWC0000027756,NaN,NaN,NaN,NaN,ALWC0000027756,0,CENTRALHUB_BLPFM,CENTRALHUB_L_DEL2,SATELLITEHUB_FKKRKL,2026-11-23,Undelivered_Not_Attended,ORD-0075987,2026-12-04 23:59:59,2026-11-22 12:21:17,NaN,614809,NON-FBF,Forward,Furniture,True,NaN,NACL Profex Super 1LTR,NaN,0.0,P4,ext_sps,ALW,ALWC0000027756,2026-11-22,2026-11-22T18:01:21.000+05:30,0.0,NaN,2026-12-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,At XD,2026-11-23,0,False,At XD,11.0,Future LPD,South,TAMIL NADU,VLG,SATELLITEHUB_FKKRKL,Ajay G Menon,NaN
78,ALWC0000027759,NaN,NaN,NaN,NaN,ALWC0000027759,0,CENTRALHUB_BLPFM,CENTRALHUB_L_BLR5,Satellitehub_RCTY,2026-11-23,Expected,ORD-0075775,2026-12-03 23:59:59,2026-11-22 11:27:47,NaN,517236,NON-FBF,Forward,Furniture,True,NaN,NACL Profex Super 500ML,NaN,0.0,P4,ext_sps,ALW,ALWC0000027759,2026-11-22,2026-11-22T18:01:21.000+05:30,0.0,NaN,2026-12-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Intransit to XD,2026-11-23,0,False,Intransit to XD,10.0,Future LPD,South,ANDHRA PRADESH,TMP,SATELLITEHUB_RCTY,Deepak Kumar,NaN
79,ALWC0000027760,NaN,NaN,NaN,NaN,ALWC0000027760,0,CENTRALHUB_BLPFM,CENTRALHUB_L_DEL2,Satellitehub_BDPR1,2026-11-23,Undelivered_Not_Attended,ORD-0075384,2026-12-01 23:59:59,2026-11-22 10:52:28,NaN,788806,NON-FBF,Forward,Furniture,True,NaN,Hymatic Charger 2 Amp Sprayer Spare Parts,NaN,0.0,P4,ext_sps,ALW,ALWC0000027760,2026-11-22,2026-11-22T18:01:25.000+05:30,0.0,NaN,2026-12-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,At XD,2026-11-23,0,False,At XD,8.0,Future LPD,East,Assam,BDP,Satellitehub_BDPR1,Suranjit Chatterjee,NaN
80,ALWC0000027762,NaN,NaN,NaN,NaN,ALWC0000027762,0,CENTRALHUB_KNL1,CENTRALHUB_L_DEL2,SatelliteHub_VNS,2026-11-23,Expected,ORD-0076113,2026-12-02 23:59:59,2026-11-21 23:42:14,NaN,231306,NON-FBF,Forward,Furniture,True,NaN,Shriram Fiprid 100G,NaN,0.0,P4,ext_sps,ALW,ALWC0000027762,2026-11-22,2026-11-22T11:42:49.000+05:30,0.0,NaN,2026-12-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Intransit to XD,2026-11-23,0,False,Intransit to XD,9.0,Future LPD,North,UTTAR PRADESH,RJA,SATELLITEHUB_VNS,DILIP KUMAR,NaN
81,ALWC0000027764,NaN,NaN,NaN,NaN,ALWC0000027764,0,CENTRALHUB_SY_RJK,CENTRALHUB_L_AMD3,Satellitehub_FKGJB,2026-11-23,Expected,ORD-0075827,2026-12-06 23:59:59,2026-11-21 19:41:00,NaN,464331,NON-FBF,Forward,Furniture,True,NaN,Star Agri Solutions Star Claim 500G,NaN,0.0,P4,ext_sps,ALW,ALWC0000027764,2026-11-22,2026-11-22T15:32:39.000+05:30,0.0,NaN,2026-12-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Intransit to XD,2026-11-23,0,False,Intransit to XD,13.0,Future LPD,West,Madhya Pradesh,GYA,Satellitehub_FKGJB,Kishore Asrani,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62652,ZHBP0000000061,NaN,NaN,NaN,NaN,ZHBP0000000061,0,CENTRALHUB_STV1,CENTRALHUB_L_MUMX,SatelliteHub_GUW,2026-11-23,Undelivered_Not_Attended,639444461130,2026-11-30 23:59:59,2026-11-20 14:11:37,NaN,781035,NON-FBF,Forward,Furniture,True,NaN,Medical Equipment,NaN,0.0,P4,ext_sps,ZHB,ZHBP0000000061,2026-11-21,2026-11-21T15:53:12.000+05:30,0.0,NaN,2026-11-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,At XD,2026-11-23,0,False,At XD,7.0,Future LPD,East,Assam,LKH,Satellitehub_G

In [22]:
df10["Freq"] = ""
df10["Hub Check"] = ""
df10["Brand Type"] = ""

In [23]:
df10

,ShipmentId,CurrentHubCity,consignmentId,ConsignmentCreationTime,ConsignmentCreationHub,parentId,MPS_count,OriginHub,CurrentHubName,DestinationHub,LastUpdateTime,LatestShipmentStatus,OrderId,CPD,ShipmentCreationTime,CustomerCity,CustomerPincode,Shipment Movement,service_type,ShipmentCategory,isMPS,isPrexo,ProductDescription,OBDType,PriorityScore,PriorityScoreV2,ItemType,merchant_code,tracking_id,ph_in_date_final,ph_in_time,fsd_number_of_ofd_attempts,dh_receive_time,shipped_lpd,AWB,Remarks,Tracking Id,Latest Status,Delivery Hub,Current Location,Latest Update Time,First Receive Time,Logistics Promise Date,No. of Attempts,remark_1,todays_date,Last_Scan_ageing,Hub_check,Remark,chek_col,LPD,Zone,State,Beat,Hub Name,GM,check,Freq,Hub Check,Brand Type
77,ALWC0000027756,NaN,NaN,NaN,NaN,ALWC0000027756,0,CENTRALHUB_BLPFM,CENTRALHUB_L_DEL2,SATELLITEHUB_FKKRKL,2026-11-23,Undelivered_Not_Attended,ORD-0075987,2026-12-04 23:59:59,2026-11-22 12:21:17,NaN,614809,NON-FBF,Forward,Furniture,True,NaN,NACL Profex Super 1LTR,NaN,0.0,P4,ext_sps,ALW,ALWC0000027756,2026-11-22,2026-11-22T18:01:21.000+05:30,0.0,NaN,2026-12-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,At XD,2026-11-23,0,False,At XD,11.0,Future LPD,South,TAMIL NADU,VLG,SATELLITEHUB_FKKRKL,Ajay G Menon,NaN,,,
78,ALWC0000027759,NaN,NaN,NaN,NaN,ALWC0000027759,0,CENTRALHUB_BLPFM,CENTRALHUB_L_BLR5,Satellitehub_RCTY,2026-11-23,Expected,ORD-0075775,2026-12-03 23:59:59,2026-11-22 11:27:47,NaN,517236,NON-FBF,Forward,Furniture,True,NaN,NACL Profex Super 500ML,NaN,0.0,P4,ext_sps,ALW,ALWC0000027759,2026-11-22,2026-11-22T18:01:21.000+05:30,0.0,NaN,2026-12-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Intransit to XD,2026-11-23,0,False,Intransit to XD,10.0,Future LPD,South,ANDHRA PRADESH,TMP,SATELLITEHUB_RCTY,Deepak Kumar,NaN,,,
79,ALWC0000027760,NaN,NaN,NaN,NaN,ALWC0000027760,0,CENTRALHUB_BLPFM,CENTRALHUB_L_DEL2,Satellitehub_BDPR1,2026-11-23,Undelivered_Not_Attended,ORD-0075384,2026-12-01 23:59:59,2026-11-22 10:52:28,NaN,788806,NON-FBF,Forward,Furniture,True,NaN,Hymatic Charger 2 Amp Sprayer Spare Parts,NaN,0.0,P4,ext_sps,ALW,ALWC0000027760,2026-11-22,2026-11-22T18:01:25.000+05:30,0.0,NaN,2026-12-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,At XD,2026-11-23,0,False,At XD,8.0,Future LPD,East,Assam,BDP,Satellitehub_BDPR1,Suranjit Chatterjee,NaN,,,
80,ALWC0000027762,NaN,NaN,NaN,NaN,ALWC0000027762,0,CENTRALHUB_KNL1,CENTRALHUB_L_DEL2,SatelliteHub_VNS,2026-11-23,Expected,ORD-0076113,2026-12-02 23:59:59,2026-11-21 23:42:14,NaN,231306,NON-FBF,Forward,Furniture,True,NaN,Shriram Fiprid 100G,NaN,0.0,P4,ext_sps,ALW,ALWC0000027762,2026-11-22,2026-11-22T11:42:49.000+05:30,0.0,NaN,2026-12-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Intransit to XD,2026-11-23,0,False,Intransit to XD,9.0,Future LPD,North,UTTAR PRADESH,RJA,SATELLITEHUB_VNS,DILIP KUMAR,NaN,,,
81,ALWC0000027764,NaN,NaN,NaN,NaN,ALWC0000027764,0,CENTRALHUB_SY_RJK,CENTRALHUB_L_AMD3,Satellitehub_FKGJB,2026-11-23,Expected,ORD-0075827,2026-12-06 23:59:59,2026-11-21 19:41:00,NaN,464331,NON-FBF,Forward,Furniture,True,NaN,Star Agri Solutions Star Claim 500G,NaN,0.0,P4,ext_sps,ALW,ALWC0000027764,2026-11-22,2026-11-22T15:32:39.000+05:30,0.0,NaN,2026-12-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Intransit to XD,2026-11-23,0,False,Intransit to XD,13.0,Future LPD,West,Madhya Pradesh,GYA,Satellitehub_FKGJB,Kishore Asrani,NaN,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62652,ZHBP0000000061,NaN,NaN,NaN,NaN,ZHBP0000000061,0,CENTRALHUB_STV1,CENTRALHUB_L_MUMX,SatelliteHub_GUW,2026-11-23,Undelivered_Not_Attended,639444461130,2026-11-30 23:59:59,2026-11-20 14:11:37,NaN,781035,NON-FBF,Forward,Furniture,True,NaN,Medical Equipment,NaN,0.0,P4,ext_sps,ZHB,ZHBP0000000061,2026-11-21,2026-11-21T15:53:12.000+05:30,0.0,NaN,2026-11-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,At XD,2026-11-23,0,Fal

In [24]:
# Remove all rows where ShipmentId starts with 'MLA'
df10 = df10[~df10['ShipmentId'].astype(str).str.startswith('MLA', na=False)]

print("✅ Successfully removed all MLA client shipments!")

✅ Successfully removed all MLA client shipments!


In [25]:
df10

,ShipmentId,CurrentHubCity,consignmentId,ConsignmentCreationTime,ConsignmentCreationHub,parentId,MPS_count,OriginHub,CurrentHubName,DestinationHub,LastUpdateTime,LatestShipmentStatus,OrderId,CPD,ShipmentCreationTime,CustomerCity,CustomerPincode,Shipment Movement,service_type,ShipmentCategory,isMPS,isPrexo,ProductDescription,OBDType,PriorityScore,PriorityScoreV2,ItemType,merchant_code,tracking_id,ph_in_date_final,ph_in_time,fsd_number_of_ofd_attempts,dh_receive_time,shipped_lpd,AWB,Remarks,Tracking Id,Latest Status,Delivery Hub,Current Location,Latest Update Time,First Receive Time,Logistics Promise Date,No. of Attempts,remark_1,todays_date,Last_Scan_ageing,Hub_check,Remark,chek_col,LPD,Zone,State,Beat,Hub Name,GM,check,Freq,Hub Check,Brand Type
77,ALWC0000027756,NaN,NaN,NaN,NaN,ALWC0000027756,0,CENTRALHUB_BLPFM,CENTRALHUB_L_DEL2,SATELLITEHUB_FKKRKL,2026-11-23,Undelivered_Not_Attended,ORD-0075987,2026-12-04 23:59:59,2026-11-22 12:21:17,NaN,614809,NON-FBF,Forward,Furniture,True,NaN,NACL Profex Super 1LTR,NaN,0.0,P4,ext_sps,ALW,ALWC0000027756,2026-11-22,2026-11-22T18:01:21.000+05:30,0.0,NaN,2026-12-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,At XD,2026-11-23,0,False,At XD,11.0,Future LPD,South,TAMIL NADU,VLG,SATELLITEHUB_FKKRKL,Ajay G Menon,NaN,,,
78,ALWC0000027759,NaN,NaN,NaN,NaN,ALWC0000027759,0,CENTRALHUB_BLPFM,CENTRALHUB_L_BLR5,Satellitehub_RCTY,2026-11-23,Expected,ORD-0075775,2026-12-03 23:59:59,2026-11-22 11:27:47,NaN,517236,NON-FBF,Forward,Furniture,True,NaN,NACL Profex Super 500ML,NaN,0.0,P4,ext_sps,ALW,ALWC0000027759,2026-11-22,2026-11-22T18:01:21.000+05:30,0.0,NaN,2026-12-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Intransit to XD,2026-11-23,0,False,Intransit to XD,10.0,Future LPD,South,ANDHRA PRADESH,TMP,SATELLITEHUB_RCTY,Deepak Kumar,NaN,,,
79,ALWC0000027760,NaN,NaN,NaN,NaN,ALWC0000027760,0,CENTRALHUB_BLPFM,CENTRALHUB_L_DEL2,Satellitehub_BDPR1,2026-11-23,Undelivered_Not_Attended,ORD-0075384,2026-12-01 23:59:59,2026-11-22 10:52:28,NaN,788806,NON-FBF,Forward,Furniture,True,NaN,Hymatic Charger 2 Amp Sprayer Spare Parts,NaN,0.0,P4,ext_sps,ALW,ALWC0000027760,2026-11-22,2026-11-22T18:01:25.000+05:30,0.0,NaN,2026-12-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,At XD,2026-11-23,0,False,At XD,8.0,Future LPD,East,Assam,BDP,Satellitehub_BDPR1,Suranjit Chatterjee,NaN,,,
80,ALWC0000027762,NaN,NaN,NaN,NaN,ALWC0000027762,0,CENTRALHUB_KNL1,CENTRALHUB_L_DEL2,SatelliteHub_VNS,2026-11-23,Expected,ORD-0076113,2026-12-02 23:59:59,2026-11-21 23:42:14,NaN,231306,NON-FBF,Forward,Furniture,True,NaN,Shriram Fiprid 100G,NaN,0.0,P4,ext_sps,ALW,ALWC0000027762,2026-11-22,2026-11-22T11:42:49.000+05:30,0.0,NaN,2026-12-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Intransit to XD,2026-11-23,0,False,Intransit to XD,9.0,Future LPD,North,UTTAR PRADESH,RJA,SATELLITEHUB_VNS,DILIP KUMAR,NaN,,,
81,ALWC0000027764,NaN,NaN,NaN,NaN,ALWC0000027764,0,CENTRALHUB_SY_RJK,CENTRALHUB_L_AMD3,Satellitehub_FKGJB,2026-11-23,Expected,ORD-0075827,2026-12-06 23:59:59,2026-11-21 19:41:00,NaN,464331,NON-FBF,Forward,Furniture,True,NaN,Star Agri Solutions Star Claim 500G,NaN,0.0,P4,ext_sps,ALW,ALWC0000027764,2026-11-22,2026-11-22T15:32:39.000+05:30,0.0,NaN,2026-12-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Intransit to XD,2026-11-23,0,False,Intransit to XD,13.0,Future LPD,West,Madhya Pradesh,GYA,Satellitehub_FKGJB,Kishore Asrani,NaN,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62652,ZHBP0000000061,NaN,NaN,NaN,NaN,ZHBP0000000061,0,CENTRALHUB_STV1,CENTRALHUB_L_MUMX,SatelliteHub_GUW,2026-11-23,Undelivered_Not_Attended,639444461130,2026-11-30 23:59:59,2026-11-20 14:11:37,NaN,781035,NON-FBF,Forward,Furniture,True,NaN,Medical Equipment,NaN,0.0,P4,ext_sps,ZHB,ZHBP0000000061,2026-11-21,2026-11-21T15:53:12.000+05:30,0.0,NaN,2026-11-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,At XD,2026-11-23,0,Fal

In [26]:
from datetime import datetime
import pandas as pd

# 1. Read the sheet from row 3 onwards (capturing your manually filtered data)
mapping_df = pd.read_excel(
    "slotsRcReport_21_Nov.xlsx", sheet_name="Sheet1", skiprows=2
)

# Clean up column names and strip spaces safely
mapping_df.columns = [str(col).strip() for col in mapping_df.columns]

# Ensure pivot columns are clean, completely UPPERCASE strings for safe linking
mapping_df["routeCode"] = mapping_df["routeCode"].astype(str).str.strip().str.upper()
mapping_df["Sum of maxCapacity"] = pd.to_numeric(mapping_df["Sum of maxCapacity"], errors='coerce').fillna(0)

# ==============================================================================
# VLOOKUP LOGIC: SINGLE KEY MAPPING (RouteCode Only)
# ==============================================================================
# Drop duplicates to mimic Excel's top-down first-match VLOOKUP behavior
vlookup_df = mapping_df.drop_duplicates(subset=["routeCode"], keep="first")

# Map tracking key pairing: routeCode -> Capacity
capacity_map = dict(
    zip(
        vlookup_df["routeCode"], 
        vlookup_df["Sum of maxCapacity"]
    )
)
# ==============================================================================

# 2. Clean up your deployment dataframe (df10) columns
df10["Beat"] = df10["Beat"].astype(str).str.strip().str.upper()

# 3. Map values using strictly the RouteCode/Beat key
# 🎯 FIXED: Changed .fillna(0) to .fillna("#N/A") to perfectly mirror Excel's error string
df10["Freq"] = df10["Beat"].map(capacity_map).fillna("#N/A")

print("🎯 Fixed! Casing differences resolved. Missing lookups will now show #N/A exactly like Excel.")

🎯 Fixed! Casing differences resolved. Missing lookups will now show #N/A exactly like Excel.


In [27]:
df10

,ShipmentId,CurrentHubCity,consignmentId,ConsignmentCreationTime,ConsignmentCreationHub,parentId,MPS_count,OriginHub,CurrentHubName,DestinationHub,LastUpdateTime,LatestShipmentStatus,OrderId,CPD,ShipmentCreationTime,CustomerCity,CustomerPincode,Shipment Movement,service_type,ShipmentCategory,isMPS,isPrexo,ProductDescription,OBDType,PriorityScore,PriorityScoreV2,ItemType,merchant_code,tracking_id,ph_in_date_final,ph_in_time,fsd_number_of_ofd_attempts,dh_receive_time,shipped_lpd,AWB,Remarks,Tracking Id,Latest Status,Delivery Hub,Current Location,Latest Update Time,First Receive Time,Logistics Promise Date,No. of Attempts,remark_1,todays_date,Last_Scan_ageing,Hub_check,Remark,chek_col,LPD,Zone,State,Beat,Hub Name,GM,check,Freq,Hub Check,Brand Type
77,ALWC0000027756,NaN,NaN,NaN,NaN,ALWC0000027756,0,CENTRALHUB_BLPFM,CENTRALHUB_L_DEL2,SATELLITEHUB_FKKRKL,2026-11-23,Undelivered_Not_Attended,ORD-0075987,2026-12-04 23:59:59,2026-11-22 12:21:17,NaN,614809,NON-FBF,Forward,Furniture,True,NaN,NACL Profex Super 1LTR,NaN,0.0,P4,ext_sps,ALW,ALWC0000027756,2026-11-22,2026-11-22T18:01:21.000+05:30,0.0,NaN,2026-12-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,At XD,2026-11-23,0,False,At XD,11.0,Future LPD,South,TAMIL NADU,VLG,SATELLITEHUB_FKKRKL,Ajay G Menon,NaN,0.0,,
78,ALWC0000027759,NaN,NaN,NaN,NaN,ALWC0000027759,0,CENTRALHUB_BLPFM,CENTRALHUB_L_BLR5,Satellitehub_RCTY,2026-11-23,Expected,ORD-0075775,2026-12-03 23:59:59,2026-11-22 11:27:47,NaN,517236,NON-FBF,Forward,Furniture,True,NaN,NACL Profex Super 500ML,NaN,0.0,P4,ext_sps,ALW,ALWC0000027759,2026-11-22,2026-11-22T18:01:21.000+05:30,0.0,NaN,2026-12-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Intransit to XD,2026-11-23,0,False,Intransit to XD,10.0,Future LPD,South,ANDHRA PRADESH,TMP,SATELLITEHUB_RCTY,Deepak Kumar,NaN,0.0,,
79,ALWC0000027760,NaN,NaN,NaN,NaN,ALWC0000027760,0,CENTRALHUB_BLPFM,CENTRALHUB_L_DEL2,Satellitehub_BDPR1,2026-11-23,Undelivered_Not_Attended,ORD-0075384,2026-12-01 23:59:59,2026-11-22 10:52:28,NaN,788806,NON-FBF,Forward,Furniture,True,NaN,Hymatic Charger 2 Amp Sprayer Spare Parts,NaN,0.0,P4,ext_sps,ALW,ALWC0000027760,2026-11-22,2026-11-22T18:01:25.000+05:30,0.0,NaN,2026-12-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,At XD,2026-11-23,0,False,At XD,8.0,Future LPD,East,Assam,BDP,Satellitehub_BDPR1,Suranjit Chatterjee,NaN,57.0,,
80,ALWC0000027762,NaN,NaN,NaN,NaN,ALWC0000027762,0,CENTRALHUB_KNL1,CENTRALHUB_L_DEL2,SatelliteHub_VNS,2026-11-23,Expected,ORD-0076113,2026-12-02 23:59:59,2026-11-21 23:42:14,NaN,231306,NON-FBF,Forward,Furniture,True,NaN,Shriram Fiprid 100G,NaN,0.0,P4,ext_sps,ALW,ALWC0000027762,2026-11-22,2026-11-22T11:42:49.000+05:30,0.0,NaN,2026-12-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Intransit to XD,2026-11-23,0,False,Intransit to XD,9.0,Future LPD,North,UTTAR PRADESH,RJA,SATELLITEHUB_VNS,DILIP KUMAR,NaN,0.0,,
81,ALWC0000027764,NaN,NaN,NaN,NaN,ALWC0000027764,0,CENTRALHUB_SY_RJK,CENTRALHUB_L_AMD3,Satellitehub_FKGJB,2026-11-23,Expected,ORD-0075827,2026-12-06 23:59:59,2026-11-21 19:41:00,NaN,464331,NON-FBF,Forward,Furniture,True,NaN,Star Agri Solutions Star Claim 500G,NaN,0.0,P4,ext_sps,ALW,ALWC0000027764,2026-11-22,2026-11-22T15:32:39.000+05:30,0.0,NaN,2026-12-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Intransit to XD,2026-11-23,0,False,Intransit to XD,13.0,Future LPD,West,Madhya Pradesh,GYA,Satellitehub_FKGJB,Kishore Asrani,NaN,0.0,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62652,ZHBP0000000061,NaN,NaN,NaN,NaN,ZHBP0000000061,0,CENTRALHUB_STV1,CENTRALHUB_L_MUMX,SatelliteHub_GUW,2026-11-23,Undelivered_Not_Attended,639444461130,2026-11-30 23:59:59,2026-11-20 14:11:37,NaN,781035,NON-FBF,Forward,Furniture,True,NaN,Medical Equipment,NaN,0.0,P4,ext_sps,ZHB,ZHBP0000000061,2026-11-21,2026-11-21T15:53:12.000+05:30,0.0,NaN,2026-11-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,At XD,

In [28]:
import numpy as np

# Apply the conditions:
# 1. If Freq is 0 -> "Beat Not Available"
# 2. If Freq is NaN/Missing (#N/A) -> "Beat Available"
# 3. If Freq > 0 -> "Beat Available"
df10["Hub Check"] = np.where(
    df10["Freq"] == 0,
    "Beat Not Available",
    "Beat Available",  # Handles both values > 0 and missing #N/A mappings
)

In [29]:
df10

,ShipmentId,CurrentHubCity,consignmentId,ConsignmentCreationTime,ConsignmentCreationHub,parentId,MPS_count,OriginHub,CurrentHubName,DestinationHub,LastUpdateTime,LatestShipmentStatus,OrderId,CPD,ShipmentCreationTime,CustomerCity,CustomerPincode,Shipment Movement,service_type,ShipmentCategory,isMPS,isPrexo,ProductDescription,OBDType,PriorityScore,PriorityScoreV2,ItemType,merchant_code,tracking_id,ph_in_date_final,ph_in_time,fsd_number_of_ofd_attempts,dh_receive_time,shipped_lpd,AWB,Remarks,Tracking Id,Latest Status,Delivery Hub,Current Location,Latest Update Time,First Receive Time,Logistics Promise Date,No. of Attempts,remark_1,todays_date,Last_Scan_ageing,Hub_check,Remark,chek_col,LPD,Zone,State,Beat,Hub Name,GM,check,Freq,Hub Check,Brand Type
77,ALWC0000027756,NaN,NaN,NaN,NaN,ALWC0000027756,0,CENTRALHUB_BLPFM,CENTRALHUB_L_DEL2,SATELLITEHUB_FKKRKL,2026-11-23,Undelivered_Not_Attended,ORD-0075987,2026-12-04 23:59:59,2026-11-22 12:21:17,NaN,614809,NON-FBF,Forward,Furniture,True,NaN,NACL Profex Super 1LTR,NaN,0.0,P4,ext_sps,ALW,ALWC0000027756,2026-11-22,2026-11-22T18:01:21.000+05:30,0.0,NaN,2026-12-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,At XD,2026-11-23,0,False,At XD,11.0,Future LPD,South,TAMIL NADU,VLG,SATELLITEHUB_FKKRKL,Ajay G Menon,NaN,0.0,Beat Not Available,
78,ALWC0000027759,NaN,NaN,NaN,NaN,ALWC0000027759,0,CENTRALHUB_BLPFM,CENTRALHUB_L_BLR5,Satellitehub_RCTY,2026-11-23,Expected,ORD-0075775,2026-12-03 23:59:59,2026-11-22 11:27:47,NaN,517236,NON-FBF,Forward,Furniture,True,NaN,NACL Profex Super 500ML,NaN,0.0,P4,ext_sps,ALW,ALWC0000027759,2026-11-22,2026-11-22T18:01:21.000+05:30,0.0,NaN,2026-12-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Intransit to XD,2026-11-23,0,False,Intransit to XD,10.0,Future LPD,South,ANDHRA PRADESH,TMP,SATELLITEHUB_RCTY,Deepak Kumar,NaN,0.0,Beat Not Available,
79,ALWC0000027760,NaN,NaN,NaN,NaN,ALWC0000027760,0,CENTRALHUB_BLPFM,CENTRALHUB_L_DEL2,Satellitehub_BDPR1,2026-11-23,Undelivered_Not_Attended,ORD-0075384,2026-12-01 23:59:59,2026-11-22 10:52:28,NaN,788806,NON-FBF,Forward,Furniture,True,NaN,Hymatic Charger 2 Amp Sprayer Spare Parts,NaN,0.0,P4,ext_sps,ALW,ALWC0000027760,2026-11-22,2026-11-22T18:01:25.000+05:30,0.0,NaN,2026-12-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,At XD,2026-11-23,0,False,At XD,8.0,Future LPD,East,Assam,BDP,Satellitehub_BDPR1,Suranjit Chatterjee,NaN,57.0,Beat Available,
80,ALWC0000027762,NaN,NaN,NaN,NaN,ALWC0000027762,0,CENTRALHUB_KNL1,CENTRALHUB_L_DEL2,SatelliteHub_VNS,2026-11-23,Expected,ORD-0076113,2026-12-02 23:59:59,2026-11-21 23:42:14,NaN,231306,NON-FBF,Forward,Furniture,True,NaN,Shriram Fiprid 100G,NaN,0.0,P4,ext_sps,ALW,ALWC0000027762,2026-11-22,2026-11-22T11:42:49.000+05:30,0.0,NaN,2026-12-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Intransit to XD,2026-11-23,0,False,Intransit to XD,9.0,Future LPD,North,UTTAR PRADESH,RJA,SATELLITEHUB_VNS,DILIP KUMAR,NaN,0.0,Beat Not Available,
81,ALWC0000027764,NaN,NaN,NaN,NaN,ALWC0000027764,0,CENTRALHUB_SY_RJK,CENTRALHUB_L_AMD3,Satellitehub_FKGJB,2026-11-23,Expected,ORD-0075827,2026-12-06 23:59:59,2026-11-21 19:41:00,NaN,464331,NON-FBF,Forward,Furniture,True,NaN,Star Agri Solutions Star Claim 500G,NaN,0.0,P4,ext_sps,ALW,ALWC0000027764,2026-11-22,2026-11-22T15:32:39.000+05:30,0.0,NaN,2026-12-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Intransit to XD,2026-11-23,0,False,Intransit to XD,13.0,Future LPD,West,Madhya Pradesh,GYA,Satellitehub_FKGJB,Kishore Asrani,NaN,0.0,Beat Not Available,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62652,ZHBP0000000061,NaN,NaN,NaN,NaN,ZHBP0000000061,0,CENTRALHUB_STV1,CENTRALHUB_L_MUMX,SatelliteHub_GUW,2026-11-23,Undelivered_Not_Attended,639444461130,2026-11-30 23:59:59,2026-11-20 14:11:37,NaN,781035,NON-FBF,Forward,Furniture,True,NaN,Medical Equipment,NaN,0.0,P4,ext_sps,ZHB,ZHBP0000000061,2026-11-21,2026-11-2

In [30]:
import pandas as pd

# 1. Read the separate brands mapping file
brand_df = pd.read_excel("Brands sheet (1).xlsx", sheet_name="Sheet1")

# Clean up column names and string properties safely
brand_df.columns = [str(col).strip() for col in brand_df.columns]

# The first column (Column A in Excel, usually 'Row Labels') contains the lookup keys
brand_df["Row Labels"] = brand_df["Row Labels"].astype(str).str.strip()

# Set up column AB name from your main data (df10) to match against
lookup_col_name = (
    "merchant_code"  # ⚠️ Change this if column AB has a different header
)
df10[lookup_col_name] = df10[lookup_col_name].astype(str).str.strip()

# 2. Get the name of the 2nd column in the brand sheet to pull values from
second_col_name = brand_df.columns[1]
brand_map = dict(zip(brand_df["Row Labels"], brand_df[second_col_name]))

# 3. Apply the VLOOKUP mapping to the 'Brand Type' column
df10["Brand Type"] = df10[lookup_col_name].map(brand_map)

# 4. Fill missing mappings or literal "#N/A" strings with 'Non Brands'
df10["Brand Type"] = df10["Brand Type"].fillna("Non Brands")
df10["Brand Type"] = df10["Brand Type"].replace("#N/A", "Non Brands")

print("✅ Brand Type successfully mapped and cleaned!")

✅ Brand Type successfully mapped and cleaned!


In [31]:
df10

,ShipmentId,CurrentHubCity,consignmentId,ConsignmentCreationTime,ConsignmentCreationHub,parentId,MPS_count,OriginHub,CurrentHubName,DestinationHub,LastUpdateTime,LatestShipmentStatus,OrderId,CPD,ShipmentCreationTime,CustomerCity,CustomerPincode,Shipment Movement,service_type,ShipmentCategory,isMPS,isPrexo,ProductDescription,OBDType,PriorityScore,PriorityScoreV2,ItemType,merchant_code,tracking_id,ph_in_date_final,ph_in_time,fsd_number_of_ofd_attempts,dh_receive_time,shipped_lpd,AWB,Remarks,Tracking Id,Latest Status,Delivery Hub,Current Location,Latest Update Time,First Receive Time,Logistics Promise Date,No. of Attempts,remark_1,todays_date,Last_Scan_ageing,Hub_check,Remark,chek_col,LPD,Zone,State,Beat,Hub Name,GM,check,Freq,Hub Check,Brand Type
77,ALWC0000027756,NaN,NaN,NaN,NaN,ALWC0000027756,0,CENTRALHUB_BLPFM,CENTRALHUB_L_DEL2,SATELLITEHUB_FKKRKL,2026-11-23,Undelivered_Not_Attended,ORD-0075987,2026-12-04 23:59:59,2026-11-22 12:21:17,NaN,614809,NON-FBF,Forward,Furniture,True,NaN,NACL Profex Super 1LTR,NaN,0.0,P4,ext_sps,ALW,ALWC0000027756,2026-11-22,2026-11-22T18:01:21.000+05:30,0.0,NaN,2026-12-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,At XD,2026-11-23,0,False,At XD,11.0,Future LPD,South,TAMIL NADU,VLG,SATELLITEHUB_FKKRKL,Ajay G Menon,NaN,0.0,Beat Not Available,Non Brands
78,ALWC0000027759,NaN,NaN,NaN,NaN,ALWC0000027759,0,CENTRALHUB_BLPFM,CENTRALHUB_L_BLR5,Satellitehub_RCTY,2026-11-23,Expected,ORD-0075775,2026-12-03 23:59:59,2026-11-22 11:27:47,NaN,517236,NON-FBF,Forward,Furniture,True,NaN,NACL Profex Super 500ML,NaN,0.0,P4,ext_sps,ALW,ALWC0000027759,2026-11-22,2026-11-22T18:01:21.000+05:30,0.0,NaN,2026-12-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Intransit to XD,2026-11-23,0,False,Intransit to XD,10.0,Future LPD,South,ANDHRA PRADESH,TMP,SATELLITEHUB_RCTY,Deepak Kumar,NaN,0.0,Beat Not Available,Non Brands
79,ALWC0000027760,NaN,NaN,NaN,NaN,ALWC0000027760,0,CENTRALHUB_BLPFM,CENTRALHUB_L_DEL2,Satellitehub_BDPR1,2026-11-23,Undelivered_Not_Attended,ORD-0075384,2026-12-01 23:59:59,2026-11-22 10:52:28,NaN,788806,NON-FBF,Forward,Furniture,True,NaN,Hymatic Charger 2 Amp Sprayer Spare Parts,NaN,0.0,P4,ext_sps,ALW,ALWC0000027760,2026-11-22,2026-11-22T18:01:25.000+05:30,0.0,NaN,2026-12-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,At XD,2026-11-23,0,False,At XD,8.0,Future LPD,East,Assam,BDP,Satellitehub_BDPR1,Suranjit Chatterjee,NaN,57.0,Beat Available,Non Brands
80,ALWC0000027762,NaN,NaN,NaN,NaN,ALWC0000027762,0,CENTRALHUB_KNL1,CENTRALHUB_L_DEL2,SatelliteHub_VNS,2026-11-23,Expected,ORD-0076113,2026-12-02 23:59:59,2026-11-21 23:42:14,NaN,231306,NON-FBF,Forward,Furniture,True,NaN,Shriram Fiprid 100G,NaN,0.0,P4,ext_sps,ALW,ALWC0000027762,2026-11-22,2026-11-22T11:42:49.000+05:30,0.0,NaN,2026-12-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Intransit to XD,2026-11-23,0,False,Intransit to XD,9.0,Future LPD,North,UTTAR PRADESH,RJA,SATELLITEHUB_VNS,DILIP KUMAR,NaN,0.0,Beat Not Available,Non Brands
81,ALWC0000027764,NaN,NaN,NaN,NaN,ALWC0000027764,0,CENTRALHUB_SY_RJK,CENTRALHUB_L_AMD3,Satellitehub_FKGJB,2026-11-23,Expected,ORD-0075827,2026-12-06 23:59:59,2026-11-21 19:41:00,NaN,464331,NON-FBF,Forward,Furniture,True,NaN,Star Agri Solutions Star Claim 500G,NaN,0.0,P4,ext_sps,ALW,ALWC0000027764,2026-11-22,2026-11-22T15:32:39.000+05:30,0.0,NaN,2026-12-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Intransit to XD,2026-11-23,0,False,Intransit to XD,13.0,Future LPD,West,Madhya Pradesh,GYA,Satellitehub_FKGJB,Kishore Asrani,NaN,0.0,Beat Not Available,Non Brands
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62652,ZHBP0000000061,NaN,NaN,NaN,NaN,ZHBP0000000061,0,CENTRALHUB_STV1,CENTRALHUB_L_MUMX,SatelliteHub_GUW,2026-11-23,Undelivered_Not_Attended,639444461130,2026-11-30 23:59:59,2026-11-20 14:11:37,NaN,781035,NON-FBF,Forward,Furniture,True,NaN,Medical Equipment,NaN,0.0,

In [32]:
import pandas as pd

# 1. Parse 'shipped_lpd' strictly as Day-First (DD-MM-YYYY) so 31-07-2026 is preserved
shipped_lpd_dt = pd.to_datetime(df10["shipped_lpd"], dayfirst=True, errors="coerce")

# 2. Calculate LPD Ageing using normalized today's date
today = pd.Timestamp.now().normalize()
df10["LPD Ageing"] = (today - shipped_lpd_dt).dt.days

# 3. Cleanly format shipped_lpd back to YYYY-MM-DD (or DD-MM-YYYY) without dropping ANY dates
df10["shipped_lpd"] = shipped_lpd_dt.dt.strftime("%d-%m-%Y")

# 4. Reposition 'LPD Ageing' right after 'shipped_lpd'
if "LPD Ageing" in df10.columns:
    lpd_ageing_col = df10.pop("LPD Ageing")
    shipped_lpd_idx = df10.columns.get_loc("shipped_lpd")
    df10.insert(shipped_lpd_idx + 1, "LPD Ageing", lpd_ageing_col)

print("✅ Perfect! 31-07-2026 and all other dates are 100% preserved.")

✅ Perfect! 31-07-2026 and all other dates are 100% preserved.


In [33]:
df10['shipped_lpd'].value_counts()

shipped_lpd
28-11-2026    6967
27-11-2026    6964
26-11-2026    6199
25-11-2026    5377
29-11-2026    5197
              ... 
19-09-2026       1
10-12-2026       1
18-09-2026       1
27-09-2026       1
16-09-2026       1
Name: count, Length: 76, dtype: int64

In [34]:
df10

,ShipmentId,CurrentHubCity,consignmentId,ConsignmentCreationTime,ConsignmentCreationHub,parentId,MPS_count,OriginHub,CurrentHubName,DestinationHub,LastUpdateTime,LatestShipmentStatus,OrderId,CPD,ShipmentCreationTime,CustomerCity,CustomerPincode,Shipment Movement,service_type,ShipmentCategory,isMPS,isPrexo,ProductDescription,OBDType,PriorityScore,PriorityScoreV2,ItemType,merchant_code,tracking_id,ph_in_date_final,ph_in_time,fsd_number_of_ofd_attempts,dh_receive_time,shipped_lpd,LPD Ageing,AWB,Remarks,Tracking Id,Latest Status,Delivery Hub,Current Location,Latest Update Time,First Receive Time,Logistics Promise Date,No. of Attempts,remark_1,todays_date,Last_Scan_ageing,Hub_check,Remark,chek_col,LPD,Zone,State,Beat,Hub Name,GM,check,Freq,Hub Check,Brand Type
77,ALWC0000027756,NaN,NaN,NaN,NaN,ALWC0000027756,0,CENTRALHUB_BLPFM,CENTRALHUB_L_DEL2,SATELLITEHUB_FKKRKL,2026-11-23,Undelivered_Not_Attended,ORD-0075987,2026-12-04 23:59:59,2026-11-22 12:21:17,NaN,614809,NON-FBF,Forward,Furniture,True,NaN,NACL Profex Super 1LTR,NaN,0.0,P4,ext_sps,ALW,ALWC0000027756,2026-11-22,2026-11-22T18:01:21.000+05:30,0.0,NaN,04-12-2026,-11.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,At XD,2026-11-23,0,False,At XD,11.0,Future LPD,South,TAMIL NADU,VLG,SATELLITEHUB_FKKRKL,Ajay G Menon,NaN,0.0,Beat Not Available,Non Brands
78,ALWC0000027759,NaN,NaN,NaN,NaN,ALWC0000027759,0,CENTRALHUB_BLPFM,CENTRALHUB_L_BLR5,Satellitehub_RCTY,2026-11-23,Expected,ORD-0075775,2026-12-03 23:59:59,2026-11-22 11:27:47,NaN,517236,NON-FBF,Forward,Furniture,True,NaN,NACL Profex Super 500ML,NaN,0.0,P4,ext_sps,ALW,ALWC0000027759,2026-11-22,2026-11-22T18:01:21.000+05:30,0.0,NaN,03-12-2026,-10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Intransit to XD,2026-11-23,0,False,Intransit to XD,10.0,Future LPD,South,ANDHRA PRADESH,TMP,SATELLITEHUB_RCTY,Deepak Kumar,NaN,0.0,Beat Not Available,Non Brands
79,ALWC0000027760,NaN,NaN,NaN,NaN,ALWC0000027760,0,CENTRALHUB_BLPFM,CENTRALHUB_L_DEL2,Satellitehub_BDPR1,2026-11-23,Undelivered_Not_Attended,ORD-0075384,2026-12-01 23:59:59,2026-11-22 10:52:28,NaN,788806,NON-FBF,Forward,Furniture,True,NaN,Hymatic Charger 2 Amp Sprayer Spare Parts,NaN,0.0,P4,ext_sps,ALW,ALWC0000027760,2026-11-22,2026-11-22T18:01:25.000+05:30,0.0,NaN,01-12-2026,-8.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,At XD,2026-11-23,0,False,At XD,8.0,Future LPD,East,Assam,BDP,Satellitehub_BDPR1,Suranjit Chatterjee,NaN,57.0,Beat Available,Non Brands
80,ALWC0000027762,NaN,NaN,NaN,NaN,ALWC0000027762,0,CENTRALHUB_KNL1,CENTRALHUB_L_DEL2,SatelliteHub_VNS,2026-11-23,Expected,ORD-0076113,2026-12-02 23:59:59,2026-11-21 23:42:14,NaN,231306,NON-FBF,Forward,Furniture,True,NaN,Shriram Fiprid 100G,NaN,0.0,P4,ext_sps,ALW,ALWC0000027762,2026-11-22,2026-11-22T11:42:49.000+05:30,0.0,NaN,02-12-2026,-9.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Intransit to XD,2026-11-23,0,False,Intransit to XD,9.0,Future LPD,North,UTTAR PRADESH,RJA,SATELLITEHUB_VNS,DILIP KUMAR,NaN,0.0,Beat Not Available,Non Brands
81,ALWC0000027764,NaN,NaN,NaN,NaN,ALWC0000027764,0,CENTRALHUB_SY_RJK,CENTRALHUB_L_AMD3,Satellitehub_FKGJB,2026-11-23,Expected,ORD-0075827,2026-12-06 23:59:59,2026-11-21 19:41:00,NaN,464331,NON-FBF,Forward,Furniture,True,NaN,Star Agri Solutions Star Claim 500G,NaN,0.0,P4,ext_sps,ALW,ALWC0000027764,2026-11-22,2026-11-22T15:32:39.000+05:30,0.0,NaN,06-12-2026,-13.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Intransit to XD,2026-11-23,0,False,Intransit to XD,13.0,Future LPD,West,Madhya Pradesh,GYA,Satellitehub_FKGJB,Kishore Asrani,NaN,0.0,Beat Not Available,Non Brands
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62652,ZHBP0000000061,NaN,NaN,NaN,NaN,ZHBP0000000061,0,CENTRALHUB_STV1,CENTRALHUB_L_MUMX,SatelliteHub_GUW,2026-11-23,Undelivered_Not_Attended,639444461130,2026-11-30 23:59:59,2026-11-20 14:11:37,NaN,781035,NON-FBF,Forward,Fu

In [35]:
import numpy as np
import pandas as pd

# 1. Clean missing/blank values in CPD column
cpd_clean = (
    df10["CPD"]
    .astype(str)
    .str.strip()
    .replace(
        [
            r"^\s*$",
            "nan",
            "NaN",
            "None",
            "null",
            "NaT",
            "<NA>",
            "0000-00-00",
        ],
        np.nan,
        regex=True,
    )
)

# 2. Parse CPD with dayfirst=False so 11 is recognized as Month (November)
cpd_dt = pd.to_datetime(cpd_clean, dayfirst=False, errors="coerce")

# 3. Fill remaining blanks in shipped_lpd_dt using cpd_dt
combined_lpd_dt = shipped_lpd_dt.fillna(cpd_dt)

# 4. Update 'shipped_lpd' formatted string column with filled values (DD-MM-YYYY)
df10["shipped_lpd"] = combined_lpd_dt.dt.strftime("%d-%m-%Y")

# 5. Recalculate 'LPD Ageing' for all rows
df10["LPD Ageing"] = (today - combined_lpd_dt.dt.normalize()).dt.days

print(
    "✅ Fixed! 11 is now correctly parsed as November (Month) and LPD Ageing is accurate."
)

✅ Fixed! 11 is now correctly parsed as November (Month) and LPD Ageing is accurate.


In [36]:
df10['shipped_lpd'].value_counts()

shipped_lpd
27-11-2026    7130
28-11-2026    7065
26-11-2026    6213
25-11-2026    5378
29-11-2026    5251
              ... 
25-02-2026       1
01-06-2026       1
27-09-2026       1
16-09-2026       1
15-09-2026       1
Name: count, Length: 81, dtype: int64

In [37]:
# ==============================================================================
# 🎯 ERROR-PROOF FIX: EXTRACT DATE SAFELY WITHOUT STRING SPLITTING
# ==============================================================================

# 1. Strip hidden spaces from column names to prevent errors
df10.columns = [str(col).strip() for col in df10.columns]

# 2. Treat variations of empty values in ph_in_date_final as true blanks (NaN)
df10["ph_in_date_final"] = df10["ph_in_date_final"].astype(str).str.strip()
df10["ph_in_date_final"] = df10["ph_in_date_final"].replace(['nan', 'None', '', 'NaT'], pd.NA)

# 3. Target ONLY rows where ph_in_date_final is blank
is_blank_target = df10["ph_in_date_final"].isna()

# 4. Native dynamic date parsing (Handles mixed strings, floats, and formats safely)
extracted_dates = pd.to_datetime(
    df10.loc[is_blank_target, "ph_in_time"], 
    dayfirst=True, 
    errors='coerce'
).dt.strftime("%Y-%m-%d")

# 5. Drop the cleanly parsed dates directly into the blank slots
df10.loc[is_blank_target, "ph_in_date_final"] = extracted_dates

# 6. Keep truly empty cells empty so Excel reads them cleanly as blanks
df10["ph_in_date_final"] = df10["ph_in_date_final"].fillna("")

print("🎯 Fixed! Mixed text and numbers handled. Available dates are successfully extracted.")

🎯 Fixed! Mixed text and numbers handled. Available dates are successfully extracted.


C:\Users\mohammadali1.vc\AppData\Local\Temp\ipykernel_22036\1995559157.py:16: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  extracted_dates = pd.to_datetime(


In [38]:
# # ==============================================================================
# # 🎯 FIX: FILL BLANKS IN No. of Attempts FROM fsd_number_of_ofd_attempts
# # ==============================================================================

# 1. Clean column headers to handle any hidden trailing spaces safely
df10.columns = [str(col).strip() for col in df10.columns]

# 2. Standardize text blanks or empty values in "No. of Attempts" to true missing values (NaN)
df10["No. of Attempts"] = df10["No. of Attempts"].astype(str).str.strip()
df10["No. of Attempts"] = df10["No. of Attempts"].replace(['nan', 'None', '', 'NaT'], pd.NA)

# 3. Fill the empty cells using values from the 'fsd_number_of_ofd_attempts' column
df10["No. of Attempts"] = df10["No. of Attempts"].fillna(df10["fsd_number_of_ofd_attempts"])

print("🎯 Done! Blanks in 'No. of Attempts' have been successfully populated.")

🎯 Done! Blanks in 'No. of Attempts' have been successfully populated.


In [39]:
# df10.to_csv("AAA.csv")

In [40]:
# df10 = pd.read_csv("AAA.csv")

In [41]:
df10

,ShipmentId,CurrentHubCity,consignmentId,ConsignmentCreationTime,ConsignmentCreationHub,parentId,MPS_count,OriginHub,CurrentHubName,DestinationHub,LastUpdateTime,LatestShipmentStatus,OrderId,CPD,ShipmentCreationTime,CustomerCity,CustomerPincode,Shipment Movement,service_type,ShipmentCategory,isMPS,isPrexo,ProductDescription,OBDType,PriorityScore,PriorityScoreV2,ItemType,merchant_code,tracking_id,ph_in_date_final,ph_in_time,fsd_number_of_ofd_attempts,dh_receive_time,shipped_lpd,LPD Ageing,AWB,Remarks,Tracking Id,Latest Status,Delivery Hub,Current Location,Latest Update Time,First Receive Time,Logistics Promise Date,No. of Attempts,remark_1,todays_date,Last_Scan_ageing,Hub_check,Remark,chek_col,LPD,Zone,State,Beat,Hub Name,GM,check,Freq,Hub Check,Brand Type
77,ALWC0000027756,NaN,NaN,NaN,NaN,ALWC0000027756,0,CENTRALHUB_BLPFM,CENTRALHUB_L_DEL2,SATELLITEHUB_FKKRKL,2026-11-23,Undelivered_Not_Attended,ORD-0075987,2026-12-04 23:59:59,2026-11-22 12:21:17,NaN,614809,NON-FBF,Forward,Furniture,True,NaN,NACL Profex Super 1LTR,NaN,0.0,P4,ext_sps,ALW,ALWC0000027756,2026-11-22,2026-11-22T18:01:21.000+05:30,0.0,NaN,04-12-2026,-11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,At XD,2026-11-23,0,False,At XD,11.0,Future LPD,South,TAMIL NADU,VLG,SATELLITEHUB_FKKRKL,Ajay G Menon,NaN,0.0,Beat Not Available,Non Brands
78,ALWC0000027759,NaN,NaN,NaN,NaN,ALWC0000027759,0,CENTRALHUB_BLPFM,CENTRALHUB_L_BLR5,Satellitehub_RCTY,2026-11-23,Expected,ORD-0075775,2026-12-03 23:59:59,2026-11-22 11:27:47,NaN,517236,NON-FBF,Forward,Furniture,True,NaN,NACL Profex Super 500ML,NaN,0.0,P4,ext_sps,ALW,ALWC0000027759,2026-11-22,2026-11-22T18:01:21.000+05:30,0.0,NaN,03-12-2026,-10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,Intransit to XD,2026-11-23,0,False,Intransit to XD,10.0,Future LPD,South,ANDHRA PRADESH,TMP,SATELLITEHUB_RCTY,Deepak Kumar,NaN,0.0,Beat Not Available,Non Brands
79,ALWC0000027760,NaN,NaN,NaN,NaN,ALWC0000027760,0,CENTRALHUB_BLPFM,CENTRALHUB_L_DEL2,Satellitehub_BDPR1,2026-11-23,Undelivered_Not_Attended,ORD-0075384,2026-12-01 23:59:59,2026-11-22 10:52:28,NaN,788806,NON-FBF,Forward,Furniture,True,NaN,Hymatic Charger 2 Amp Sprayer Spare Parts,NaN,0.0,P4,ext_sps,ALW,ALWC0000027760,2026-11-22,2026-11-22T18:01:25.000+05:30,0.0,NaN,01-12-2026,-8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,At XD,2026-11-23,0,False,At XD,8.0,Future LPD,East,Assam,BDP,Satellitehub_BDPR1,Suranjit Chatterjee,NaN,57.0,Beat Available,Non Brands
80,ALWC0000027762,NaN,NaN,NaN,NaN,ALWC0000027762,0,CENTRALHUB_KNL1,CENTRALHUB_L_DEL2,SatelliteHub_VNS,2026-11-23,Expected,ORD-0076113,2026-12-02 23:59:59,2026-11-21 23:42:14,NaN,231306,NON-FBF,Forward,Furniture,True,NaN,Shriram Fiprid 100G,NaN,0.0,P4,ext_sps,ALW,ALWC0000027762,2026-11-22,2026-11-22T11:42:49.000+05:30,0.0,NaN,02-12-2026,-9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,Intransit to XD,2026-11-23,0,False,Intransit to XD,9.0,Future LPD,North,UTTAR PRADESH,RJA,SATELLITEHUB_VNS,DILIP KUMAR,NaN,0.0,Beat Not Available,Non Brands
81,ALWC0000027764,NaN,NaN,NaN,NaN,ALWC0000027764,0,CENTRALHUB_SY_RJK,CENTRALHUB_L_AMD3,Satellitehub_FKGJB,2026-11-23,Expected,ORD-0075827,2026-12-06 23:59:59,2026-11-21 19:41:00,NaN,464331,NON-FBF,Forward,Furniture,True,NaN,Star Agri Solutions Star Claim 500G,NaN,0.0,P4,ext_sps,ALW,ALWC0000027764,2026-11-22,2026-11-22T15:32:39.000+05:30,0.0,NaN,06-12-2026,-13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,Intransit to XD,2026-11-23,0,False,Intransit to XD,13.0,Future LPD,West,Madhya Pradesh,GYA,Satellitehub_FKGJB,Kishore Asrani,NaN,0.0,Beat Not Available,Non Brands
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62652,ZHBP0000000061,NaN,NaN,NaN,NaN,ZHBP0000000061,0,CENTRALHUB_STV1,CENTRALHUB_L_MUMX,SatelliteHub_GUW,2026-11-23,Undelivered_Not_Attended,639444461130,2026-11-30 23:59:59,2026-11-20 14:11:37,NaN,781035,NON-FBF,Forward,Furniture,Tr

In [42]:
# # Keep only rows where remark_1 is NOT 'Not_picked'
# df10 = df10[df10['remark_1'] != 'Not_picked']

In [43]:
df10

,ShipmentId,CurrentHubCity,consignmentId,ConsignmentCreationTime,ConsignmentCreationHub,parentId,MPS_count,OriginHub,CurrentHubName,DestinationHub,LastUpdateTime,LatestShipmentStatus,OrderId,CPD,ShipmentCreationTime,CustomerCity,CustomerPincode,Shipment Movement,service_type,ShipmentCategory,isMPS,isPrexo,ProductDescription,OBDType,PriorityScore,PriorityScoreV2,ItemType,merchant_code,tracking_id,ph_in_date_final,ph_in_time,fsd_number_of_ofd_attempts,dh_receive_time,shipped_lpd,LPD Ageing,AWB,Remarks,Tracking Id,Latest Status,Delivery Hub,Current Location,Latest Update Time,First Receive Time,Logistics Promise Date,No. of Attempts,remark_1,todays_date,Last_Scan_ageing,Hub_check,Remark,chek_col,LPD,Zone,State,Beat,Hub Name,GM,check,Freq,Hub Check,Brand Type
77,ALWC0000027756,NaN,NaN,NaN,NaN,ALWC0000027756,0,CENTRALHUB_BLPFM,CENTRALHUB_L_DEL2,SATELLITEHUB_FKKRKL,2026-11-23,Undelivered_Not_Attended,ORD-0075987,2026-12-04 23:59:59,2026-11-22 12:21:17,NaN,614809,NON-FBF,Forward,Furniture,True,NaN,NACL Profex Super 1LTR,NaN,0.0,P4,ext_sps,ALW,ALWC0000027756,2026-11-22,2026-11-22T18:01:21.000+05:30,0.0,NaN,04-12-2026,-11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,At XD,2026-11-23,0,False,At XD,11.0,Future LPD,South,TAMIL NADU,VLG,SATELLITEHUB_FKKRKL,Ajay G Menon,NaN,0.0,Beat Not Available,Non Brands
78,ALWC0000027759,NaN,NaN,NaN,NaN,ALWC0000027759,0,CENTRALHUB_BLPFM,CENTRALHUB_L_BLR5,Satellitehub_RCTY,2026-11-23,Expected,ORD-0075775,2026-12-03 23:59:59,2026-11-22 11:27:47,NaN,517236,NON-FBF,Forward,Furniture,True,NaN,NACL Profex Super 500ML,NaN,0.0,P4,ext_sps,ALW,ALWC0000027759,2026-11-22,2026-11-22T18:01:21.000+05:30,0.0,NaN,03-12-2026,-10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,Intransit to XD,2026-11-23,0,False,Intransit to XD,10.0,Future LPD,South,ANDHRA PRADESH,TMP,SATELLITEHUB_RCTY,Deepak Kumar,NaN,0.0,Beat Not Available,Non Brands
79,ALWC0000027760,NaN,NaN,NaN,NaN,ALWC0000027760,0,CENTRALHUB_BLPFM,CENTRALHUB_L_DEL2,Satellitehub_BDPR1,2026-11-23,Undelivered_Not_Attended,ORD-0075384,2026-12-01 23:59:59,2026-11-22 10:52:28,NaN,788806,NON-FBF,Forward,Furniture,True,NaN,Hymatic Charger 2 Amp Sprayer Spare Parts,NaN,0.0,P4,ext_sps,ALW,ALWC0000027760,2026-11-22,2026-11-22T18:01:25.000+05:30,0.0,NaN,01-12-2026,-8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,At XD,2026-11-23,0,False,At XD,8.0,Future LPD,East,Assam,BDP,Satellitehub_BDPR1,Suranjit Chatterjee,NaN,57.0,Beat Available,Non Brands
80,ALWC0000027762,NaN,NaN,NaN,NaN,ALWC0000027762,0,CENTRALHUB_KNL1,CENTRALHUB_L_DEL2,SatelliteHub_VNS,2026-11-23,Expected,ORD-0076113,2026-12-02 23:59:59,2026-11-21 23:42:14,NaN,231306,NON-FBF,Forward,Furniture,True,NaN,Shriram Fiprid 100G,NaN,0.0,P4,ext_sps,ALW,ALWC0000027762,2026-11-22,2026-11-22T11:42:49.000+05:30,0.0,NaN,02-12-2026,-9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,Intransit to XD,2026-11-23,0,False,Intransit to XD,9.0,Future LPD,North,UTTAR PRADESH,RJA,SATELLITEHUB_VNS,DILIP KUMAR,NaN,0.0,Beat Not Available,Non Brands
81,ALWC0000027764,NaN,NaN,NaN,NaN,ALWC0000027764,0,CENTRALHUB_SY_RJK,CENTRALHUB_L_AMD3,Satellitehub_FKGJB,2026-11-23,Expected,ORD-0075827,2026-12-06 23:59:59,2026-11-21 19:41:00,NaN,464331,NON-FBF,Forward,Furniture,True,NaN,Star Agri Solutions Star Claim 500G,NaN,0.0,P4,ext_sps,ALW,ALWC0000027764,2026-11-22,2026-11-22T15:32:39.000+05:30,0.0,NaN,06-12-2026,-13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,Intransit to XD,2026-11-23,0,False,Intransit to XD,13.0,Future LPD,West,Madhya Pradesh,GYA,Satellitehub_FKGJB,Kishore Asrani,NaN,0.0,Beat Not Available,Non Brands
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62652,ZHBP0000000061,NaN,NaN,NaN,NaN,ZHBP0000000061,0,CENTRALHUB_STV1,CENTRALHUB_L_MUMX,SatelliteHub_GUW,2026-11-23,Undelivered_Not_Attended,639444461130,2026-11-30 23:59:59,2026-11-20 14:11:37,NaN,781035,NON-FBF,Forward,Furniture,Tr

In [44]:
import pandas as pd

# 1. Filter data where 'check' is 'Blank'
df_filtered = df10[df10["check"].astype(str).str.strip().str.lower() == "blank"]

# 2. Create the Pivot Table
pivot_1 = pd.pivot_table(
    df_filtered,
    values="ShipmentId",
    index="GM",
    columns="remark_1",
    aggfunc="count",
    fill_value="",  # Matches Excel blanks
    margins=True,
    margins_name="Grand Total",
)

# 3. 🔢 SORT BY GRAND TOTAL (Descending, keeping Grand Total row at the bottom)
# Separate the 'Grand Total' row from the rest of the data to sort properly
data_rows = pivot_1.iloc[:-1]
grand_total_row = pivot_1.iloc[[-1]]

# Sort the data rows by the 'Grand Total' column from largest to smallest
data_rows_sorted = data_rows.sort_values(by="Freshlanding", ascending=False)

# Combine them back together
pivot_1_sorted = pd.concat([data_rows_sorted, grand_total_row])

# 4. Clean headers for Excel-like look
pivot_1_sorted.index.name = "GM"
pivot_1_sorted.columns.name = ""

# Show the sorted pivot table
pivot_1_sorted

C:\Users\mohammadali1.vc\AppData\Local\Temp\ipykernel_22036\2182819501.py:7: Pandas4Warning: Using a fill_value that cannot be held in the existing dtype is deprecated and will raise in a future version.
  pivot_1 = pd.pivot_table(


,Freshlanding,NCD>48hrs,NDR-1,NDR-2,NSS,OSM,SSM,Grand Total
GM,,,,,,,,
DILIP KUMAR,1697,173,158,51,,,,2079
Rakesh Kumar,1506,98,220,59,,,,1883
Ajay G Menon,1401,120,146,55,,,,1722
Kishore Asrani,1321,166,112,28,,4,,1631
Biplob Ranjan Dhar,1155,85,167,55,,,,1462
Suranjit Chatterjee,1063,54,130,33,7,,,1287
Sankarshan Mukherjee,1021,62,114,21,,,,1218
Dayanidhi G,985,76,122,29,,8,,1220
Ateeque Akhtar Faridi,770,139,157,39,,15,1,1121


In [45]:
df10[["shipped_lpd","LPD Ageing"]]

,shipped_lpd,LPD Ageing
77,04-12-2026,-11
78,03-12-2026,-10
79,01-12-2026,-8
80,02-12-2026,-9
81,06-12-2026,-13
...,...,...
62652,30-11-2026,-7
62653,03-12-2026,-10
62654,28-11-2026,-5
62655,28-11-2026,-5


In [46]:
import pandas as pd

# 1. Filter data where 'check' is 'Blank' (case-insensitive)
df_filtered = df10[df10["check"].astype(str).str.strip().str.lower() == "blank"]

# 2. Create Pivot Table 2 (State vs remark_1)
pivot_2 = pd.pivot_table(
    df_filtered,
    values="ShipmentId",
    index="State",  # Rows use State
    columns="remark_1",  # Columns use remark_1 categories
    aggfunc="count",
    fill_value="",  # Clear blank look matching Excel
    margins=True,
    margins_name="Grand Total",
)

# 3. 🔢 SORT BY GRAND TOTAL (Keeping 'Grand Total' row at the bottom)
data_rows_2 = pivot_2.iloc[:-1]
grand_total_row_2 = pivot_2.iloc[[-1]]

# Sort data rows from largest to smallest volume
data_rows_sorted_2 = data_rows_2.sort_values(by="Grand Total", ascending=False)

# Combine sorted rows and total row back together
pivot_2_sorted = pd.concat([data_rows_sorted_2, grand_total_row_2])

# 4. Format headers cleanly like Excel
pivot_2_sorted.index.name = "State"
pivot_2_sorted.columns.name = ""

# Show the second sorted pivot table
pivot_2_sorted

C:\Users\mohammadali1.vc\AppData\Local\Temp\ipykernel_22036\2562820727.py:7: Pandas4Warning: Using a fill_value that cannot be held in the existing dtype is deprecated and will raise in a future version.
  pivot_2 = pd.pivot_table(


,Freshlanding,NCD>48hrs,NDR-1,NDR-2,NSS,OSM,SSM,Grand Total
State,,,,,,,,
UTTAR PRADESH,1856,209,177,60,,,,2302
Bihar,1506,98,220,59,,,,1883
West Bengal,1067,78,145,46,,,,1336
TAMIL NADU,1013,137,121,33,,,,1304
Gujarat,936,117,81,19,,4,,1157
KARNATAKA,914,77,117,28,,8,,1144
Maharashtra,770,139,157,39,,15,1,1121
KERALA,828,16,80,36,,,,960
ANDHRA PRADESH,465,93,97,32,,,,687


In [47]:
df10.drop(columns=["chek_col"],inplace=True)

In [48]:
df10.drop(columns=["LPD"],inplace=True)

In [49]:
import numpy as np
import pandas as pd

# 1. Ensure 'LPD' and 'chek_col' are removed if they already exist
for col in ["chek_col", "LPD"]:
    if col in df10.columns:
        df10.drop(columns=[col], inplace=True)

# 2. Assign updated LPD status directly based on 'LPD Ageing'
df10["LPD"] = np.where(
    df10["LPD Ageing"] < 0,
    "Future LPD",
    np.where(
        df10["LPD Ageing"] == 0,
        "Todays LPD",
        np.where(df10["LPD Ageing"] > 0, "Past LPD", None),
    ),
)

# 3. Position 'LPD' directly after 'LPD Ageing'
if "LPD Ageing" in df10.columns:
    lpd_col = df10.pop("LPD")
    lpd_ageing_idx = df10.columns.get_loc("LPD Ageing")
    df10.insert(lpd_ageing_idx + 1, "LPD", lpd_col)

print("✅ LPD column successfully updated and inserted right after LPD Ageing!")

✅ LPD column successfully updated and inserted right after LPD Ageing!


In [50]:
df10['LPD'].value_counts()

LPD
Future LPD    44050
Past LPD       4178
Todays LPD     1950
Name: count, dtype: int64

In [51]:
import pandas as pd

# 1. Prepare data with clean '(blank)' placeholders
df_pivot3 = df10.copy()
df_pivot3["remark_1"] = df_pivot3["remark_1"].fillna("(blank)").astype(str).str.strip()
df_pivot3["LPD"] = df_pivot3["LPD"].fillna("(blank)").astype(str).str.strip()

df_pivot3["remark_1"] = df_pivot3["remark_1"].replace("", "(blank)")
df_pivot3["LPD"] = df_pivot3["LPD"].replace("", "(blank)")

# 2. Generate the baseline pivot table
pivot_3 = pd.pivot_table(
    df_pivot3,
    values="ShipmentId",
    index="remark_1",
    columns="LPD",
    aggfunc="count",
    fill_value="",
    margins=True,
    margins_name="Grand Total",
)

# 3. 🔢 SORT ROWS BY GRAND TOTAL (Keeping 'Grand Total' row at the bottom)
data_rows_3 = pivot_3.iloc[:-1]
grand_total_row_3 = pivot_3.iloc[[-1]]
data_rows_sorted_3 = data_rows_3.sort_values(by="Grand Total", ascending=False)
pivot_3_final = pd.concat([data_rows_sorted_3, grand_total_row_3])

# 4. 🧭 REORDER COLUMNS TO MATCH EXCEL LAYOUT
# We explicitly define the order: Main categories -> (blank) -> Grand Total
desired_column_order = ["Future LPD", "Past LPD", "Todays LPD", "(blank)", "Grand Total"]

# Filter out any columns that might missing in your specific data chunk to prevent errors
existing_columns = [col for col in desired_column_order if col in pivot_3_final.columns]
pivot_3_final = pivot_3_final[existing_columns]

# 5. Clean up label headers
pivot_3_final.index.name = "Row Labels"
pivot_3_final.columns.name = ""

# Show your perfectly arranged table!
pivot_3_final

C:\Users\mohammadali1.vc\AppData\Local\Temp\ipykernel_22036\2571546313.py:12: Pandas4Warning: Using a fill_value that cannot be held in the existing dtype is deprecated and will raise in a future version.
  pivot_3 = pd.pivot_table(


,Future LPD,Past LPD,Todays LPD,Grand Total
Row Labels,,,,
Intransit to XD,17870,426,122,18418
Freshlanding,11970,578,1188,13736
At XD,10171,557,157,10885
Intransit to DH,2095,135,37,2267
NDR-1,789,740,240,1769
NCD>48hrs,684,515,135,1334
>3attempts,39,802,14,855
NDR-2,109,368,46,523
AT FM,286,18,3,307


In [52]:
import pandas as pd

# 1. Filter data where 'check' is 'Blank' (case-insensitive)
df_filtered = df10[df10["check"].astype(str).str.strip().str.lower() == "blank"]

# 2. Create Pivot Table 4
pivot_4 = pd.pivot_table(
    df_filtered,
    values="ShipmentId",
    index="remark_1",  # Rows use remark_1 categories
    aggfunc="count",
    fill_value=0,
    margins=True,
    margins_name="Grand Total",
)

# 3. 🔢 SORT BY GRAND TOTAL (Keeping 'Grand Total' row pinned at the bottom)
data_rows_4 = pivot_4.iloc[:-1]
grand_total_row_4 = pivot_4.iloc[[-1]]

# Sort data rows from largest to smallest volume
data_rows_sorted_4 = data_rows_4.sort_values(by="ShipmentId", ascending=False)

# Combine sorted rows and total row back together
pivot_4_sorted = pd.concat([data_rows_sorted_4, grand_total_row_4])

# 4. Format headers cleanly like Excel
pivot_4_sorted.index.name = "Row Labels"
pivot_4_sorted.columns = ["No. of Shipments"]

# Show the fourth sorted pivot table
pivot_4_sorted

,No. of Shipments
Row Labels,
Freshlanding,13736
NDR-1,1769
NCD>48hrs,1334
NDR-2,523
OSM,27
NSS,8
SSM,1
Grand Total,17398


In [53]:
import pandas as pd

# 1. Fill any real missing values in the Zone column with '(blank)' just in case
df_pivot5 = df10.copy()
df_pivot5["Zone"] = df_pivot5["Zone"].fillna("(blank)").astype(str).str.strip()

# 2. Create Pivot Table 5 (Zone vs Count of ShipmentId)
pivot_5 = pd.pivot_table(
    df_pivot5,
    values="ShipmentId",
    index="Zone",  # Rows use Zone
    aggfunc="count",
    margins=True,
    margins_name="Grand Total",
)

# 3. 🔢 SORT BY GRAND TOTAL (Keeping 'Grand Total' row pinned at the bottom)
data_rows_5 = pivot_5.iloc[:-1]
grand_total_row_5 = pivot_5.iloc[[-1]]

# Sort data rows from largest to smallest volume
data_rows_sorted_5 = data_rows_5.sort_values(by="ShipmentId", ascending=False)

# Combine sorted rows and total row back together
pivot_5_sorted = pd.concat([data_rows_sorted_5, grand_total_row_5])

# 4. Format headers cleanly like Excel
pivot_5_sorted.index.name = "Zone"
pivot_5_sorted.columns = ["No. of Shipments"]

# Show the final beautiful 5th pivot table
pivot_5_sorted

,No. of Shipments
Zone,
East,17329
South,13430
North,11617
West,7802
Grand Total,50178


In [54]:
# df10.to_csv("BBB.csv",index=False)

In [55]:
# import pandas as pd

# # 1. Start with a fresh copy of df10
# df_pivot6 = df10.copy()

# # 2. Apply all structural rules from your Excel screenshots
# # A. Filter LPD: Keep only 'Past LPD' and 'Todays LPD'
# df_pivot6 = df_pivot6[df_pivot6["LPD"].str.strip().isin(["Past LPD", "Todays LPD"])]

# # B. Filter remark_1: Explicitly include ONLY your 7 checked categories
# allowed_remarks = [
#     "At XD",
#     "Freshlanding",
#     "Intransit to DH",
#     "Intransit to XD",
#     "NCD>48hrs",
#     "NDR-1",
#     "NDR-2"
# ]
# df_pivot6 = df_pivot6[df_pivot6["remark_1"].str.strip().isin(allowed_remarks)]

# # C. Filter LPD Ageing: Force numbers and limit to 0 through 21
# df_pivot6["LPD Ageing"] = pd.to_numeric(df_pivot6["LPD Ageing"], errors="coerce")
# df_pivot6 = df_pivot6[(df_pivot6["LPD Ageing"] >= 0) & (df_pivot6["LPD Ageing"] <= 21)]

# # 3. Build the Pivot Table
# pivot_6 = pd.pivot_table(
#     df_pivot6,
#     values="ShipmentId",
#     index="CurrentHubName",      # Rows
#     columns="LPD Ageing",  # Columns (0 to 21)
#     aggfunc="count",
#     fill_value="",         # Blanks for empty cell intersections
#     margins=True,
#     margins_name="Grand Total",
# )

# # 4. Sort Hub Name rows alphabetically (Grand Total remains at the bottom)
# data_rows_6 = pivot_6.iloc[:-1].sort_index()
# grand_total_row_6 = pivot_6.iloc[[-1]]
# pivot_6_final = pd.concat([data_rows_6, grand_total_row_6])

# # 5. Apply header names matching Excel layout
# pivot_6_final.index.name = "Row Labels"
# pivot_6_final.columns.name = "Column Labels"

# # Preview the exact configuration match
# pivot_6_final.head()

In [56]:
df10

,ShipmentId,CurrentHubCity,consignmentId,ConsignmentCreationTime,ConsignmentCreationHub,parentId,MPS_count,OriginHub,CurrentHubName,DestinationHub,LastUpdateTime,LatestShipmentStatus,OrderId,CPD,ShipmentCreationTime,CustomerCity,CustomerPincode,Shipment Movement,service_type,ShipmentCategory,isMPS,isPrexo,ProductDescription,OBDType,PriorityScore,PriorityScoreV2,ItemType,merchant_code,tracking_id,ph_in_date_final,ph_in_time,fsd_number_of_ofd_attempts,dh_receive_time,shipped_lpd,LPD Ageing,LPD,AWB,Remarks,Tracking Id,Latest Status,Delivery Hub,Current Location,Latest Update Time,First Receive Time,Logistics Promise Date,No. of Attempts,remark_1,todays_date,Last_Scan_ageing,Hub_check,Remark,Zone,State,Beat,Hub Name,GM,check,Freq,Hub Check,Brand Type
77,ALWC0000027756,NaN,NaN,NaN,NaN,ALWC0000027756,0,CENTRALHUB_BLPFM,CENTRALHUB_L_DEL2,SATELLITEHUB_FKKRKL,2026-11-23,Undelivered_Not_Attended,ORD-0075987,2026-12-04 23:59:59,2026-11-22 12:21:17,NaN,614809,NON-FBF,Forward,Furniture,True,NaN,NACL Profex Super 1LTR,NaN,0.0,P4,ext_sps,ALW,ALWC0000027756,2026-11-22,2026-11-22T18:01:21.000+05:30,0.0,NaN,04-12-2026,-11,Future LPD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,At XD,2026-11-23,0,False,At XD,South,TAMIL NADU,VLG,SATELLITEHUB_FKKRKL,Ajay G Menon,NaN,0.0,Beat Not Available,Non Brands
78,ALWC0000027759,NaN,NaN,NaN,NaN,ALWC0000027759,0,CENTRALHUB_BLPFM,CENTRALHUB_L_BLR5,Satellitehub_RCTY,2026-11-23,Expected,ORD-0075775,2026-12-03 23:59:59,2026-11-22 11:27:47,NaN,517236,NON-FBF,Forward,Furniture,True,NaN,NACL Profex Super 500ML,NaN,0.0,P4,ext_sps,ALW,ALWC0000027759,2026-11-22,2026-11-22T18:01:21.000+05:30,0.0,NaN,03-12-2026,-10,Future LPD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,Intransit to XD,2026-11-23,0,False,Intransit to XD,South,ANDHRA PRADESH,TMP,SATELLITEHUB_RCTY,Deepak Kumar,NaN,0.0,Beat Not Available,Non Brands
79,ALWC0000027760,NaN,NaN,NaN,NaN,ALWC0000027760,0,CENTRALHUB_BLPFM,CENTRALHUB_L_DEL2,Satellitehub_BDPR1,2026-11-23,Undelivered_Not_Attended,ORD-0075384,2026-12-01 23:59:59,2026-11-22 10:52:28,NaN,788806,NON-FBF,Forward,Furniture,True,NaN,Hymatic Charger 2 Amp Sprayer Spare Parts,NaN,0.0,P4,ext_sps,ALW,ALWC0000027760,2026-11-22,2026-11-22T18:01:25.000+05:30,0.0,NaN,01-12-2026,-8,Future LPD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,At XD,2026-11-23,0,False,At XD,East,Assam,BDP,Satellitehub_BDPR1,Suranjit Chatterjee,NaN,57.0,Beat Available,Non Brands
80,ALWC0000027762,NaN,NaN,NaN,NaN,ALWC0000027762,0,CENTRALHUB_KNL1,CENTRALHUB_L_DEL2,SatelliteHub_VNS,2026-11-23,Expected,ORD-0076113,2026-12-02 23:59:59,2026-11-21 23:42:14,NaN,231306,NON-FBF,Forward,Furniture,True,NaN,Shriram Fiprid 100G,NaN,0.0,P4,ext_sps,ALW,ALWC0000027762,2026-11-22,2026-11-22T11:42:49.000+05:30,0.0,NaN,02-12-2026,-9,Future LPD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,Intransit to XD,2026-11-23,0,False,Intransit to XD,North,UTTAR PRADESH,RJA,SATELLITEHUB_VNS,DILIP KUMAR,NaN,0.0,Beat Not Available,Non Brands
81,ALWC0000027764,NaN,NaN,NaN,NaN,ALWC0000027764,0,CENTRALHUB_SY_RJK,CENTRALHUB_L_AMD3,Satellitehub_FKGJB,2026-11-23,Expected,ORD-0075827,2026-12-06 23:59:59,2026-11-21 19:41:00,NaN,464331,NON-FBF,Forward,Furniture,True,NaN,Star Agri Solutions Star Claim 500G,NaN,0.0,P4,ext_sps,ALW,ALWC0000027764,2026-11-22,2026-11-22T15:32:39.000+05:30,0.0,NaN,06-12-2026,-13,Future LPD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,Intransit to XD,2026-11-23,0,False,Intransit to XD,West,Madhya Pradesh,GYA,Satellitehub_FKGJB,Kishore Asrani,NaN,0.0,Beat Not Available,Non Brands
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62652,ZHBP0000000061,NaN,NaN,NaN,NaN,ZHBP0000000061,0,CENTRALHUB_STV1,CENTRALHUB_L_MUMX,SatelliteHub_GUW,2026-11-23,Undelivered_Not_Attended,639444461130,2026-11-30 23:59:59,2026-11-20 14:11:37,NaN,781035,NON-FBF,Forward,Furniture,True,NaN,Medical Equipment,NaN,0.0,P4,

In [57]:
# df10['CPD'] = pd.to_datetime(df10['CPD'], dayfirst=True, format="mixed", errors='coerce').dt.strftime('%Y-%m-%d')

In [58]:
df10

,ShipmentId,CurrentHubCity,consignmentId,ConsignmentCreationTime,ConsignmentCreationHub,parentId,MPS_count,OriginHub,CurrentHubName,DestinationHub,LastUpdateTime,LatestShipmentStatus,OrderId,CPD,ShipmentCreationTime,CustomerCity,CustomerPincode,Shipment Movement,service_type,ShipmentCategory,isMPS,isPrexo,ProductDescription,OBDType,PriorityScore,PriorityScoreV2,ItemType,merchant_code,tracking_id,ph_in_date_final,ph_in_time,fsd_number_of_ofd_attempts,dh_receive_time,shipped_lpd,LPD Ageing,LPD,AWB,Remarks,Tracking Id,Latest Status,Delivery Hub,Current Location,Latest Update Time,First Receive Time,Logistics Promise Date,No. of Attempts,remark_1,todays_date,Last_Scan_ageing,Hub_check,Remark,Zone,State,Beat,Hub Name,GM,check,Freq,Hub Check,Brand Type
77,ALWC0000027756,NaN,NaN,NaN,NaN,ALWC0000027756,0,CENTRALHUB_BLPFM,CENTRALHUB_L_DEL2,SATELLITEHUB_FKKRKL,2026-11-23,Undelivered_Not_Attended,ORD-0075987,2026-12-04 23:59:59,2026-11-22 12:21:17,NaN,614809,NON-FBF,Forward,Furniture,True,NaN,NACL Profex Super 1LTR,NaN,0.0,P4,ext_sps,ALW,ALWC0000027756,2026-11-22,2026-11-22T18:01:21.000+05:30,0.0,NaN,04-12-2026,-11,Future LPD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,At XD,2026-11-23,0,False,At XD,South,TAMIL NADU,VLG,SATELLITEHUB_FKKRKL,Ajay G Menon,NaN,0.0,Beat Not Available,Non Brands
78,ALWC0000027759,NaN,NaN,NaN,NaN,ALWC0000027759,0,CENTRALHUB_BLPFM,CENTRALHUB_L_BLR5,Satellitehub_RCTY,2026-11-23,Expected,ORD-0075775,2026-12-03 23:59:59,2026-11-22 11:27:47,NaN,517236,NON-FBF,Forward,Furniture,True,NaN,NACL Profex Super 500ML,NaN,0.0,P4,ext_sps,ALW,ALWC0000027759,2026-11-22,2026-11-22T18:01:21.000+05:30,0.0,NaN,03-12-2026,-10,Future LPD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,Intransit to XD,2026-11-23,0,False,Intransit to XD,South,ANDHRA PRADESH,TMP,SATELLITEHUB_RCTY,Deepak Kumar,NaN,0.0,Beat Not Available,Non Brands
79,ALWC0000027760,NaN,NaN,NaN,NaN,ALWC0000027760,0,CENTRALHUB_BLPFM,CENTRALHUB_L_DEL2,Satellitehub_BDPR1,2026-11-23,Undelivered_Not_Attended,ORD-0075384,2026-12-01 23:59:59,2026-11-22 10:52:28,NaN,788806,NON-FBF,Forward,Furniture,True,NaN,Hymatic Charger 2 Amp Sprayer Spare Parts,NaN,0.0,P4,ext_sps,ALW,ALWC0000027760,2026-11-22,2026-11-22T18:01:25.000+05:30,0.0,NaN,01-12-2026,-8,Future LPD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,At XD,2026-11-23,0,False,At XD,East,Assam,BDP,Satellitehub_BDPR1,Suranjit Chatterjee,NaN,57.0,Beat Available,Non Brands
80,ALWC0000027762,NaN,NaN,NaN,NaN,ALWC0000027762,0,CENTRALHUB_KNL1,CENTRALHUB_L_DEL2,SatelliteHub_VNS,2026-11-23,Expected,ORD-0076113,2026-12-02 23:59:59,2026-11-21 23:42:14,NaN,231306,NON-FBF,Forward,Furniture,True,NaN,Shriram Fiprid 100G,NaN,0.0,P4,ext_sps,ALW,ALWC0000027762,2026-11-22,2026-11-22T11:42:49.000+05:30,0.0,NaN,02-12-2026,-9,Future LPD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,Intransit to XD,2026-11-23,0,False,Intransit to XD,North,UTTAR PRADESH,RJA,SATELLITEHUB_VNS,DILIP KUMAR,NaN,0.0,Beat Not Available,Non Brands
81,ALWC0000027764,NaN,NaN,NaN,NaN,ALWC0000027764,0,CENTRALHUB_SY_RJK,CENTRALHUB_L_AMD3,Satellitehub_FKGJB,2026-11-23,Expected,ORD-0075827,2026-12-06 23:59:59,2026-11-21 19:41:00,NaN,464331,NON-FBF,Forward,Furniture,True,NaN,Star Agri Solutions Star Claim 500G,NaN,0.0,P4,ext_sps,ALW,ALWC0000027764,2026-11-22,2026-11-22T15:32:39.000+05:30,0.0,NaN,06-12-2026,-13,Future LPD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,Intransit to XD,2026-11-23,0,False,Intransit to XD,West,Madhya Pradesh,GYA,Satellitehub_FKGJB,Kishore Asrani,NaN,0.0,Beat Not Available,Non Brands
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62652,ZHBP0000000061,NaN,NaN,NaN,NaN,ZHBP0000000061,0,CENTRALHUB_STV1,CENTRALHUB_L_MUMX,SatelliteHub_GUW,2026-11-23,Undelivered_Not_Attended,639444461130,2026-11-30 23:59:59,2026-11-20 14:11:37,NaN,781035,NON-FBF,Forward,Furniture,True,NaN,Medical Equipment,NaN,0.0,P4,

In [59]:
df10.drop(columns=["Remarks"],inplace=True)

In [60]:
df10.drop(columns=["AWB"],inplace=True)

In [61]:
df10.drop(columns=["Remark"],inplace=True)

In [62]:
df10.drop(columns=["fsd_number_of_ofd_attempts"],inplace=True)

In [63]:
df10.drop(columns=["Hub_check"],inplace=True)

In [64]:
# df10.drop(columns=["chek_col"],inplace=True)

In [65]:
# df10.drop(columns=["LPD"],inplace=True)

In [66]:
# import numpy as np
# import pandas as pd

# # 1. Ensure 'LPD' and 'chek_col' are removed if they already exist
# for col in ["chek_col", "LPD"]:
#     if col in df10.columns:
#         df10.drop(columns=[col], inplace=True)

# # 2. Assign updated LPD status directly based on 'LPD Ageing'
# df10["LPD"] = np.where(
#     df10["LPD Ageing"] < 0,
#     "Future LPD",
#     np.where(
#         df10["LPD Ageing"] == 0,
#         "Todays LPD",
#         np.where(df10["LPD Ageing"] > 0, "Past LPD", None),
#     ),
# )

# # 3. Position 'LPD' directly after 'LPD Ageing'
# if "LPD Ageing" in df10.columns:
#     lpd_col = df10.pop("LPD")
#     lpd_ageing_idx = df10.columns.get_loc("LPD Ageing")
#     df10.insert(lpd_ageing_idx + 1, "LPD", lpd_col)

# print("✅ LPD column successfully updated and inserted right after LPD Ageing!")

In [67]:
# df10['LPD'].value_counts()

In [68]:
df10

,ShipmentId,CurrentHubCity,consignmentId,ConsignmentCreationTime,ConsignmentCreationHub,parentId,MPS_count,OriginHub,CurrentHubName,DestinationHub,LastUpdateTime,LatestShipmentStatus,OrderId,CPD,ShipmentCreationTime,CustomerCity,CustomerPincode,Shipment Movement,service_type,ShipmentCategory,isMPS,isPrexo,ProductDescription,OBDType,PriorityScore,PriorityScoreV2,ItemType,merchant_code,tracking_id,ph_in_date_final,ph_in_time,dh_receive_time,shipped_lpd,LPD Ageing,LPD,Tracking Id,Latest Status,Delivery Hub,Current Location,Latest Update Time,First Receive Time,Logistics Promise Date,No. of Attempts,remark_1,todays_date,Last_Scan_ageing,Zone,State,Beat,Hub Name,GM,check,Freq,Hub Check,Brand Type
77,ALWC0000027756,NaN,NaN,NaN,NaN,ALWC0000027756,0,CENTRALHUB_BLPFM,CENTRALHUB_L_DEL2,SATELLITEHUB_FKKRKL,2026-11-23,Undelivered_Not_Attended,ORD-0075987,2026-12-04 23:59:59,2026-11-22 12:21:17,NaN,614809,NON-FBF,Forward,Furniture,True,NaN,NACL Profex Super 1LTR,NaN,0.0,P4,ext_sps,ALW,ALWC0000027756,2026-11-22,2026-11-22T18:01:21.000+05:30,NaN,04-12-2026,-11,Future LPD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,At XD,2026-11-23,0,South,TAMIL NADU,VLG,SATELLITEHUB_FKKRKL,Ajay G Menon,NaN,0.0,Beat Not Available,Non Brands
78,ALWC0000027759,NaN,NaN,NaN,NaN,ALWC0000027759,0,CENTRALHUB_BLPFM,CENTRALHUB_L_BLR5,Satellitehub_RCTY,2026-11-23,Expected,ORD-0075775,2026-12-03 23:59:59,2026-11-22 11:27:47,NaN,517236,NON-FBF,Forward,Furniture,True,NaN,NACL Profex Super 500ML,NaN,0.0,P4,ext_sps,ALW,ALWC0000027759,2026-11-22,2026-11-22T18:01:21.000+05:30,NaN,03-12-2026,-10,Future LPD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,Intransit to XD,2026-11-23,0,South,ANDHRA PRADESH,TMP,SATELLITEHUB_RCTY,Deepak Kumar,NaN,0.0,Beat Not Available,Non Brands
79,ALWC0000027760,NaN,NaN,NaN,NaN,ALWC0000027760,0,CENTRALHUB_BLPFM,CENTRALHUB_L_DEL2,Satellitehub_BDPR1,2026-11-23,Undelivered_Not_Attended,ORD-0075384,2026-12-01 23:59:59,2026-11-22 10:52:28,NaN,788806,NON-FBF,Forward,Furniture,True,NaN,Hymatic Charger 2 Amp Sprayer Spare Parts,NaN,0.0,P4,ext_sps,ALW,ALWC0000027760,2026-11-22,2026-11-22T18:01:25.000+05:30,NaN,01-12-2026,-8,Future LPD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,At XD,2026-11-23,0,East,Assam,BDP,Satellitehub_BDPR1,Suranjit Chatterjee,NaN,57.0,Beat Available,Non Brands
80,ALWC0000027762,NaN,NaN,NaN,NaN,ALWC0000027762,0,CENTRALHUB_KNL1,CENTRALHUB_L_DEL2,SatelliteHub_VNS,2026-11-23,Expected,ORD-0076113,2026-12-02 23:59:59,2026-11-21 23:42:14,NaN,231306,NON-FBF,Forward,Furniture,True,NaN,Shriram Fiprid 100G,NaN,0.0,P4,ext_sps,ALW,ALWC0000027762,2026-11-22,2026-11-22T11:42:49.000+05:30,NaN,02-12-2026,-9,Future LPD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,Intransit to XD,2026-11-23,0,North,UTTAR PRADESH,RJA,SATELLITEHUB_VNS,DILIP KUMAR,NaN,0.0,Beat Not Available,Non Brands
81,ALWC0000027764,NaN,NaN,NaN,NaN,ALWC0000027764,0,CENTRALHUB_SY_RJK,CENTRALHUB_L_AMD3,Satellitehub_FKGJB,2026-11-23,Expected,ORD-0075827,2026-12-06 23:59:59,2026-11-21 19:41:00,NaN,464331,NON-FBF,Forward,Furniture,True,NaN,Star Agri Solutions Star Claim 500G,NaN,0.0,P4,ext_sps,ALW,ALWC0000027764,2026-11-22,2026-11-22T15:32:39.000+05:30,NaN,06-12-2026,-13,Future LPD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,Intransit to XD,2026-11-23,0,West,Madhya Pradesh,GYA,Satellitehub_FKGJB,Kishore Asrani,NaN,0.0,Beat Not Available,Non Brands
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62652,ZHBP0000000061,NaN,NaN,NaN,NaN,ZHBP0000000061,0,CENTRALHUB_STV1,CENTRALHUB_L_MUMX,SatelliteHub_GUW,2026-11-23,Undelivered_Not_Attended,639444461130,2026-11-30 23:59:59,2026-11-20 14:11:37,NaN,781035,NON-FBF,Forward,Furniture,True,NaN,Medical Equipment,NaN,0.0,P4,ext_sps,ZHB,ZHBP0000000061,2026-11-21,2026-11-21T15:53:12.000+05:30,NaN,30-11-2026,-7,Future LPD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,At XD,2026-11-23,0,East,Assam,LKH,Satellitehub_GUW,Suranjit Chatterjee,NaN,44.0,Beat Available,No

In [69]:

# 2. Clean and unify ph_in_time
clean_str = df10["ph_in_time"].astype(str).str.replace(r"([+-]\d{2}:\d{2}|Z)$", "", regex=True)
parsed = pd.to_datetime(clean_str, format="ISO8601", errors="coerce")
mask = parsed.isna()
parsed.loc[mask] = pd.to_datetime(clean_str[mask], dayfirst=True, errors="coerce")

df10["ph_in_time"] = parsed

# 3. Extract date into ph_in_date_final (true Excel date without time)
df10["ph_in_date_final"] = df10["ph_in_time"].dt.date


In [70]:
print("Match counts:\n", (df10["ph_in_date_final"] == df10["ph_in_time"].dt.date).value_counts())
print("\nUnparsed (NaT) rows:", df10["ph_in_time"].isna().sum())
print("\nMismatched rows:", len(df10[df10["ph_in_date_final"] != df10["ph_in_time"].dt.date]))

Match counts:
 True    50178
Name: count, dtype: int64

Unparsed (NaT) rows: 0

Mismatched rows: 0


In [71]:
# View the 16 unparsed/mismatched rows
unparsed_rows = df10[df10["ph_in_time"].isna()]
display(unparsed_rows[["tracking_id", "ph_in_date_final", "ph_in_time"]])

,tracking_id,ph_in_date_final,ph_in_time


In [72]:
import pandas as pd

# 1. Ensure ph_in_date_final is in datetime format
df10["ph_in_date_final"] = pd.to_datetime(df10["ph_in_date_final"])

# 2. Calculate ageing against today's date (or use pd.Timestamp('2026-11-16') if fixing the date)
today = pd.Timestamp.today().normalize()
ph_in_ageing = (today - df10["ph_in_date_final"]).dt.days

# 3. Get the index position right after 'ph_in_date_final'
insert_loc = df10.columns.get_loc("ph_in_date_final") + 1

# 4. Insert the column (drop first if re-running to avoid ValueError)
if "Ph_in ageing" in df10.columns:
    df10.drop(columns=["Ph_in ageing"], inplace=True)

df10.insert(loc=insert_loc, column="Ph_in ageing", value=ph_in_ageing)

# Preview the result
df10[["ph_in_date_final", "Ph_in ageing"]].head()

,ph_in_date_final,Ph_in ageing
77,2026-11-22,1
78,2026-11-22,1
79,2026-11-22,1
80,2026-11-22,1
81,2026-11-22,1


In [73]:
df10

,ShipmentId,CurrentHubCity,consignmentId,ConsignmentCreationTime,ConsignmentCreationHub,parentId,MPS_count,OriginHub,CurrentHubName,DestinationHub,LastUpdateTime,LatestShipmentStatus,OrderId,CPD,ShipmentCreationTime,CustomerCity,CustomerPincode,Shipment Movement,service_type,ShipmentCategory,isMPS,isPrexo,ProductDescription,OBDType,PriorityScore,PriorityScoreV2,ItemType,merchant_code,tracking_id,ph_in_date_final,Ph_in ageing,ph_in_time,dh_receive_time,shipped_lpd,LPD Ageing,LPD,Tracking Id,Latest Status,Delivery Hub,Current Location,Latest Update Time,First Receive Time,Logistics Promise Date,No. of Attempts,remark_1,todays_date,Last_Scan_ageing,Zone,State,Beat,Hub Name,GM,check,Freq,Hub Check,Brand Type
77,ALWC0000027756,NaN,NaN,NaN,NaN,ALWC0000027756,0,CENTRALHUB_BLPFM,CENTRALHUB_L_DEL2,SATELLITEHUB_FKKRKL,2026-11-23,Undelivered_Not_Attended,ORD-0075987,2026-12-04 23:59:59,2026-11-22 12:21:17,NaN,614809,NON-FBF,Forward,Furniture,True,NaN,NACL Profex Super 1LTR,NaN,0.0,P4,ext_sps,ALW,ALWC0000027756,2026-11-22,1,2026-11-22 18:01:21,NaN,04-12-2026,-11,Future LPD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,At XD,2026-11-23,0,South,TAMIL NADU,VLG,SATELLITEHUB_FKKRKL,Ajay G Menon,NaN,0.0,Beat Not Available,Non Brands
78,ALWC0000027759,NaN,NaN,NaN,NaN,ALWC0000027759,0,CENTRALHUB_BLPFM,CENTRALHUB_L_BLR5,Satellitehub_RCTY,2026-11-23,Expected,ORD-0075775,2026-12-03 23:59:59,2026-11-22 11:27:47,NaN,517236,NON-FBF,Forward,Furniture,True,NaN,NACL Profex Super 500ML,NaN,0.0,P4,ext_sps,ALW,ALWC0000027759,2026-11-22,1,2026-11-22 18:01:21,NaN,03-12-2026,-10,Future LPD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,Intransit to XD,2026-11-23,0,South,ANDHRA PRADESH,TMP,SATELLITEHUB_RCTY,Deepak Kumar,NaN,0.0,Beat Not Available,Non Brands
79,ALWC0000027760,NaN,NaN,NaN,NaN,ALWC0000027760,0,CENTRALHUB_BLPFM,CENTRALHUB_L_DEL2,Satellitehub_BDPR1,2026-11-23,Undelivered_Not_Attended,ORD-0075384,2026-12-01 23:59:59,2026-11-22 10:52:28,NaN,788806,NON-FBF,Forward,Furniture,True,NaN,Hymatic Charger 2 Amp Sprayer Spare Parts,NaN,0.0,P4,ext_sps,ALW,ALWC0000027760,2026-11-22,1,2026-11-22 18:01:25,NaN,01-12-2026,-8,Future LPD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,At XD,2026-11-23,0,East,Assam,BDP,Satellitehub_BDPR1,Suranjit Chatterjee,NaN,57.0,Beat Available,Non Brands
80,ALWC0000027762,NaN,NaN,NaN,NaN,ALWC0000027762,0,CENTRALHUB_KNL1,CENTRALHUB_L_DEL2,SatelliteHub_VNS,2026-11-23,Expected,ORD-0076113,2026-12-02 23:59:59,2026-11-21 23:42:14,NaN,231306,NON-FBF,Forward,Furniture,True,NaN,Shriram Fiprid 100G,NaN,0.0,P4,ext_sps,ALW,ALWC0000027762,2026-11-22,1,2026-11-22 11:42:49,NaN,02-12-2026,-9,Future LPD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,Intransit to XD,2026-11-23,0,North,UTTAR PRADESH,RJA,SATELLITEHUB_VNS,DILIP KUMAR,NaN,0.0,Beat Not Available,Non Brands
81,ALWC0000027764,NaN,NaN,NaN,NaN,ALWC0000027764,0,CENTRALHUB_SY_RJK,CENTRALHUB_L_AMD3,Satellitehub_FKGJB,2026-11-23,Expected,ORD-0075827,2026-12-06 23:59:59,2026-11-21 19:41:00,NaN,464331,NON-FBF,Forward,Furniture,True,NaN,Star Agri Solutions Star Claim 500G,NaN,0.0,P4,ext_sps,ALW,ALWC0000027764,2026-11-22,1,2026-11-22 15:32:39,NaN,06-12-2026,-13,Future LPD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,Intransit to XD,2026-11-23,0,West,Madhya Pradesh,GYA,Satellitehub_FKGJB,Kishore Asrani,NaN,0.0,Beat Not Available,Non Brands
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62652,ZHBP0000000061,NaN,NaN,NaN,NaN,ZHBP0000000061,0,CENTRALHUB_STV1,CENTRALHUB_L_MUMX,SatelliteHub_GUW,2026-11-23,Undelivered_Not_Attended,639444461130,2026-11-30 23:59:59,2026-11-20 14:11:37,NaN,781035,NON-FBF,Forward,Furniture,True,NaN,Medical Equipment,NaN,0.0,P4,ext_sps,ZHB,ZHBP0000000061,2026-11-21,2,2026-11-21 15:53:12,NaN,30-11-2026,-7,Future LPD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,At XD,2026-11-23,0,East,Assam,LKH,Satellitehub_GUW,Suranjit Chatterjee,NaN,44.0,Beat Available,Non Brands
62653,ZHBP0000000062,N

In [74]:
import pandas as pd

# 1. Convert dh_receive_time to clean datetime (timezone-naive)
dh_dt = pd.to_datetime(df10["dh_receive_time"], errors="coerce").dt.tz_localize(None)

# 2. Format dh_receive_time column itself to match Excel (DD-MM-YYYY)
# (Note: If you want YYYY-MM-DD instead, use "%Y-%m-%d")
df10["dh_receive_time"] = dh_dt.dt.strftime("%d-%m-%Y")

# 3. Calculate DH_in Ageing using the clean date
today = pd.Timestamp.today().normalize()
dh_in_ageing = (today - dh_dt.dt.normalize()).dt.days

# 4. Insert 'DH_in Ageing' right after 'dh_receive_time'
insert_loc = df10.columns.get_loc("dh_receive_time") + 1

if "DH_in Ageing" in df10.columns:
    df10.drop(columns=["DH_in Ageing"], inplace=True)

df10.insert(loc=insert_loc, column="DH_in Ageing", value=dh_in_ageing)

# Preview the result
df10[["dh_receive_time", "DH_in Ageing"]].dropna().head()

,dh_receive_time,DH_in Ageing
98,22-11-2026,1.0
122,21-11-2026,2.0
140,23-11-2026,0.0
152,22-11-2026,1.0
153,22-11-2026,1.0


In [75]:
df10

,ShipmentId,CurrentHubCity,consignmentId,ConsignmentCreationTime,ConsignmentCreationHub,parentId,MPS_count,OriginHub,CurrentHubName,DestinationHub,LastUpdateTime,LatestShipmentStatus,OrderId,CPD,ShipmentCreationTime,CustomerCity,CustomerPincode,Shipment Movement,service_type,ShipmentCategory,isMPS,isPrexo,ProductDescription,OBDType,PriorityScore,PriorityScoreV2,ItemType,merchant_code,tracking_id,ph_in_date_final,Ph_in ageing,ph_in_time,dh_receive_time,DH_in Ageing,shipped_lpd,LPD Ageing,LPD,Tracking Id,Latest Status,Delivery Hub,Current Location,Latest Update Time,First Receive Time,Logistics Promise Date,No. of Attempts,remark_1,todays_date,Last_Scan_ageing,Zone,State,Beat,Hub Name,GM,check,Freq,Hub Check,Brand Type
77,ALWC0000027756,NaN,NaN,NaN,NaN,ALWC0000027756,0,CENTRALHUB_BLPFM,CENTRALHUB_L_DEL2,SATELLITEHUB_FKKRKL,2026-11-23,Undelivered_Not_Attended,ORD-0075987,2026-12-04 23:59:59,2026-11-22 12:21:17,NaN,614809,NON-FBF,Forward,Furniture,True,NaN,NACL Profex Super 1LTR,NaN,0.0,P4,ext_sps,ALW,ALWC0000027756,2026-11-22,1,2026-11-22 18:01:21,NaN,NaN,04-12-2026,-11,Future LPD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,At XD,2026-11-23,0,South,TAMIL NADU,VLG,SATELLITEHUB_FKKRKL,Ajay G Menon,NaN,0.0,Beat Not Available,Non Brands
78,ALWC0000027759,NaN,NaN,NaN,NaN,ALWC0000027759,0,CENTRALHUB_BLPFM,CENTRALHUB_L_BLR5,Satellitehub_RCTY,2026-11-23,Expected,ORD-0075775,2026-12-03 23:59:59,2026-11-22 11:27:47,NaN,517236,NON-FBF,Forward,Furniture,True,NaN,NACL Profex Super 500ML,NaN,0.0,P4,ext_sps,ALW,ALWC0000027759,2026-11-22,1,2026-11-22 18:01:21,NaN,NaN,03-12-2026,-10,Future LPD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,Intransit to XD,2026-11-23,0,South,ANDHRA PRADESH,TMP,SATELLITEHUB_RCTY,Deepak Kumar,NaN,0.0,Beat Not Available,Non Brands
79,ALWC0000027760,NaN,NaN,NaN,NaN,ALWC0000027760,0,CENTRALHUB_BLPFM,CENTRALHUB_L_DEL2,Satellitehub_BDPR1,2026-11-23,Undelivered_Not_Attended,ORD-0075384,2026-12-01 23:59:59,2026-11-22 10:52:28,NaN,788806,NON-FBF,Forward,Furniture,True,NaN,Hymatic Charger 2 Amp Sprayer Spare Parts,NaN,0.0,P4,ext_sps,ALW,ALWC0000027760,2026-11-22,1,2026-11-22 18:01:25,NaN,NaN,01-12-2026,-8,Future LPD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,At XD,2026-11-23,0,East,Assam,BDP,Satellitehub_BDPR1,Suranjit Chatterjee,NaN,57.0,Beat Available,Non Brands
80,ALWC0000027762,NaN,NaN,NaN,NaN,ALWC0000027762,0,CENTRALHUB_KNL1,CENTRALHUB_L_DEL2,SatelliteHub_VNS,2026-11-23,Expected,ORD-0076113,2026-12-02 23:59:59,2026-11-21 23:42:14,NaN,231306,NON-FBF,Forward,Furniture,True,NaN,Shriram Fiprid 100G,NaN,0.0,P4,ext_sps,ALW,ALWC0000027762,2026-11-22,1,2026-11-22 11:42:49,NaN,NaN,02-12-2026,-9,Future LPD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,Intransit to XD,2026-11-23,0,North,UTTAR PRADESH,RJA,SATELLITEHUB_VNS,DILIP KUMAR,NaN,0.0,Beat Not Available,Non Brands
81,ALWC0000027764,NaN,NaN,NaN,NaN,ALWC0000027764,0,CENTRALHUB_SY_RJK,CENTRALHUB_L_AMD3,Satellitehub_FKGJB,2026-11-23,Expected,ORD-0075827,2026-12-06 23:59:59,2026-11-21 19:41:00,NaN,464331,NON-FBF,Forward,Furniture,True,NaN,Star Agri Solutions Star Claim 500G,NaN,0.0,P4,ext_sps,ALW,ALWC0000027764,2026-11-22,1,2026-11-22 15:32:39,NaN,NaN,06-12-2026,-13,Future LPD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,Intransit to XD,2026-11-23,0,West,Madhya Pradesh,GYA,Satellitehub_FKGJB,Kishore Asrani,NaN,0.0,Beat Not Available,Non Brands
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62652,ZHBP0000000061,NaN,NaN,NaN,NaN,ZHBP0000000061,0,CENTRALHUB_STV1,CENTRALHUB_L_MUMX,SatelliteHub_GUW,2026-11-23,Undelivered_Not_Attended,639444461130,2026-11-30 23:59:59,2026-11-20 14:11:37,NaN,781035,NON-FBF,Forward,Furniture,True,NaN,Medical Equipment,NaN,0.0,P4,ext_sps,ZHB,ZHBP0000000061,2026-11-21,2,2026-11-21 15:53:12,NaN,NaN,30-11-2026,-7,Future LPD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,At XD,2026-11-23,0,East,Assam,LKH,Satellitehub_GUW,Suranjit Chatterjee,NaN,44.0,Beat Av

In [76]:
# df10.to_csv('dh.csv',index=False)

In [77]:
lpd_index = df10.columns.get_loc('LPD') + 1

df10.insert(
    loc=lpd_index, 
    column='new_remark', 
    value=np.where(df10['LPD'] == 'Future LPD', 'FDD', '')
)

In [78]:
df10['new_remark'].value_counts(dropna=False)

new_remark
FDD    44050
        6128
Name: count, dtype: int64

In [79]:
# Condition 1: new_remark is an empty string '' (or NaN)
is_blank = (df10['new_remark'] == '') | df10['new_remark'].isna()

# Condition 2: remark_1 is '>3attempts' (stripping extra spaces just in case)
has_3attempts = df10['remark_1'].astype(str).str.strip() == '>3attempts'

# Update new_remark where both conditions are True
df10.loc[is_blank & has_3attempts, 'new_remark'] = '>3attempts'

# Check the updated counts
df10['new_remark'].value_counts()

new_remark
FDD           44050
               5312
>3attempts      816
Name: count, dtype: int64

In [80]:
# List of the 4 remarks from remark_1
dh_remarks = ['Freshlanding', 'NCD>48hrs', 'NDR-1', 'NDR-2']

# Condition 1: new_remark is currently blank
is_blank = (df10['new_remark'] == '') | df10['new_remark'].isna()

# Condition 2: remark_1 is one of the 4 specified values
in_dh_remarks = df10['remark_1'].astype(str).str.strip().isin(dh_remarks)

# Assign 'At DH' to new_remark where both conditions are True
df10.loc[is_blank & in_dh_remarks, 'new_remark'] = 'At DH'

# Verify the updated counts
df10['new_remark'].value_counts()

new_remark
FDD           44050
At DH          3810
               1502
>3attempts      816
Name: count, dtype: int64

In [81]:
# List of the 2 remarks from remark_1
xd_remarks = ['AT FM', 'At XD']

# Condition 1: new_remark is currently blank
is_blank = (df10['new_remark'] == '') | df10['new_remark'].isna()

# Condition 2: remark_1 is one of the 2 specified values
in_xd_remarks = df10['remark_1'].astype(str).str.strip().isin(xd_remarks)

# Assign 'AT XD' to new_remark where both conditions are True
df10.loc[is_blank & in_xd_remarks, 'new_remark'] = 'AT XD'

# Verify the updated counts
df10['new_remark'].value_counts()

new_remark
FDD           44050
At DH          3810
>3attempts      816
                767
AT XD           735
Name: count, dtype: int64

In [82]:
# List of the 2 remarks from remark_1
xd_remarks = ['Intransit to DH']

# Condition 1: new_remark is currently blank
is_blank = (df10['new_remark'] == '') | df10['new_remark'].isna()

# Condition 2: remark_1 is one of the 2 specified values
in_xd_remarks = df10['remark_1'].astype(str).str.strip().isin(xd_remarks)

# Assign 'AT XD' to new_remark where both conditions are True
df10.loc[is_blank & in_xd_remarks, 'new_remark'] = 'Intransit to DH'

# Verify the updated counts
df10['new_remark'].value_counts()

new_remark
FDD                44050
At DH               3810
>3attempts           816
AT XD                735
                     595
Intransit to DH      172
Name: count, dtype: int64

In [83]:
# List of the 2 remarks from remark_1
xd_remarks = ['Intransit to XD']

# Condition 1: new_remark is currently blank
is_blank = (df10['new_remark'] == '') | df10['new_remark'].isna()

# Condition 2: remark_1 is one of the 2 specified values
in_xd_remarks = df10['remark_1'].astype(str).str.strip().isin(xd_remarks)

# Assign 'AT XD' to new_remark where both conditions are True
df10.loc[is_blank & in_xd_remarks, 'new_remark'] = 'Intransit to XD'

# Verify the updated counts
df10['new_remark'].value_counts()

new_remark
FDD                44050
At DH               3810
>3attempts           816
AT XD                735
Intransit to XD      548
Intransit to DH      172
                      47
Name: count, dtype: int64

In [84]:
# List of the 4 remarks from remark_1
nss_remarks = ['NSS', 'ORC', 'OSM', 'SSM']

# Condition 1: new_remark is currently blank
is_blank = (df10['new_remark'] == '') | df10['new_remark'].isna()

# Condition 2: remark_1 is one of the 4 specified values
in_nss_remarks = df10['remark_1'].astype(str).str.strip().isin(nss_remarks)

# Assign 'NSS/OSM/SSM/ORC' to new_remark where both conditions are True
df10.loc[is_blank & in_nss_remarks, 'new_remark'] = 'NSS/OSM/SSM/ORC'

# Verify the updated counts
df10['new_remark'].value_counts()

new_remark
FDD                44050
At DH               3810
>3attempts           816
AT XD                735
Intransit to XD      548
Intransit to DH      172
NSS/OSM/SSM/ORC       41
                       6
Name: count, dtype: int64

In [85]:
import smtplib
import pandas as pd
from datetime import datetime
from email.mime.multipart import MIMEMultipart
from email.mime.base import MIMEBase
from email.mime.text import MIMEText
from email import encoders
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

# ==============================================================================
# 1. GENERATE THE STYLED EXCEL WORKBOOK (Raw Data, Pivot, Sheet3)
# ==============================================================================
# CHANGED: Dynamically get today's date in %d %b %Y format (e.g., 24 Jun 2026)
current_date_str = datetime.today().strftime("%d %b %Y")
output_filename = f"Large FWD Pendency {current_date_str}.csv"

# header_fill = PatternFill(start_color="1F4E78", end_color="1F4E78", fill_type="solid")
# header_font = Font(name="Calibri", size=11, bold=True, color="FFFFFF")
# zebra_fill = PatternFill(start_color="F2F5F8", end_color="F2F5F8", fill_type="solid")
# total_fill = PatternFill(start_color="D9E1F2", end_color="D9E1F2", fill_type="solid")
# total_font = Font(name="Calibri", size=11, bold=True, color="000000")
# title_font = Font(name="Calibri", size=11, bold=True, color="1F4E78")
# regular_font = Font(name="Calibri", size=11, color="000000")
# thin_border = Border(
#     left=Side(style='thin', color='D9D9D9'), right=Side(style='thin', color='D9D9D9'),
#     top=Side(style='thin', color='D9D9D9'), bottom=Side(style='thin', color='D9D9D9')
# )

# def style_table_range(ws, start_row, end_row, start_col, end_col, is_summary_style=False):
#     for r in range(start_row, end_row + 1):
#         is_total_row = (r == end_row)
#         for c in range(start_col, end_col + 1):
#             cell = ws.cell(row=r, column=c)
#             cell.font = regular_font
#             cell.border = thin_border
#             if isinstance(cell.value, (int, float)):
#                 cell.alignment = Alignment(horizontal="right")
#             else:
#                 cell.alignment = Alignment(horizontal="left")
#             if r == start_row:
#                 cell.fill = header_fill
#                 cell.font = header_font
#                 cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
#             elif is_total_row:
#                 cell.fill = total_fill
#                 cell.font = total_font
#             elif r % 2 == 0 and not is_summary_style:
#                 cell.fill = zebra_fill


df10.to_csv(output_filename.replace('.xlsx', '.csv'), index=False)

#      ws2 = writer.book.create_sheet(title="Pivot")
#     current_row = 1
#     pivot_list_sheet2 = [
#         ("Table 1: GM vs remark_1 Breakdown (check: Blank)", pivot_1_sorted, False),
#         ("Table 2: State vs remark_1 Breakdown (check: Blank)", pivot_2_sorted, False),
#         ("Table 3: remark_1 vs LPD Breakdown", pivot_3_final, False),
#         ("Table 4: Total Volume by remark_1 (check: Blank)", pivot_4_sorted, True),
#         ("Table 5: Total Volume by Zone", pivot_5_sorted, True)
#     ]

#     for title, target_pivot, is_summary in pivot_list_sheet2:
#         ws2.cell(row=current_row, column=1, value=title).font = title_font
#         current_row += 1
#         target_pivot.to_excel(writer, sheet_name="Pivot", startrow=current_row - 1, startcol=0, index=True)
#         style_table_range(ws2, current_row, current_row + len(target_pivot), 1, len(target_pivot.columns) + 1, is_summary_style=is_summary)
#         current_row += len(target_pivot) + 4

#     ws3 = writer.book.create_sheet(title="Sheet3")
#     filter_header = [["check", "(All)"], ["LPD", "(Multiple Items)"], ["remark_1", "(Multiple Items)"]]
#     for idx, (lbl, val) in enumerate(filter_header):
#         ws3.cell(row=idx+1, column=1, value=lbl).font = Font(name="Calibri", size=11, bold=True)
#         ws3.cell(row=idx+1, column=2, value=val).font = regular_font
        
#     pivot_6_final.to_excel(writer, sheet_name="Sheet3", startrow=4, startcol=0, index=True)
#     style_table_range(ws3, 5, 5 + len(pivot_6_final), 1, len(pivot_6_final.columns) + 1)

#     if "Sheet" in writer.book.sheetnames:
#         writer.book.remove(writer.book["Sheet"])

print(f"✨ CSV file saved. Preparing interactive HTML email body...")

✨ CSV file saved. Preparing interactive HTML email body...


In [86]:
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.base import MIMEBase
from email.mime.text import MIMEText
from email import encoders
import pandas as pd
from datetime import datetime

# ==============================================================================
# AUTOMATED DATE LOGIC
# ==============================================================================
current_date_str = datetime.today().strftime("%d %b %Y")  # e.g., "24 Jun 2026"

# ==============================================================================
# 1. CLEAN CONVERSION FUNCTION WITH AUTOFIT WIDTHS AND DARK OUTLINES
# ==============================================================================
def generate_clean_html_table(df, row_label_name):
    html = '<table class="pivot-table">'
    html += '<thead><tr>'
    html += f'<th class="text-left">{row_label_name}</th>'
    for col in df.columns:
        html += f'<th class="text-right">{col}</th>'
    html += '</tr></thead><tbody>'
    
    for idx, row in df.iterrows():
        is_total = (str(idx).strip() == "Grand Total")
        row_class = ' class="grand-total-row"' if is_total else ''
        html += f'<tr{row_class}>'
        html += f'<th class="text-left" style="background-color: transparent; color: inherit; font-weight: {"bold" if is_total else "normal"}; border: 1px solid #555555; padding: 6px 12px; text-align: left;">{idx}</th>'
        for val in row:
            val_str = "" if pd.isna(val) or val == "" else str(val)
            html += f'<td class="text-right">{val_str}</td>'
        html += '</tr>'
        
    html += '</tbody></table>'
    return html

# ==============================================================================
# 2. EMAIL CONTAINER CSS AND MARKUP CONFIGURATION
# ==============================================================================
html_table_style = """
<style>
    body { font-family: Calibri, Arial, sans-serif; color: #333333; line-height: 1.4; }
    h3 { color: #1F4E78; margin-bottom: 8px; margin-top: 30px; font-size: 13pt; font-family: Calibri, sans-serif; }
    
    table.pivot-table { 
        border-collapse: collapse; 
        margin-bottom: 20px; 
        font-size: 10.5pt; 
        font-family: Calibri, sans-serif;
        width: auto;                  
        border: 1.5px solid #555555;   
    }
    table.pivot-table th { 
        background-color: #1F4E78; 
        color: white; 
        padding: 8px 14px; 
        border: 1px solid #555555;     
        font-weight: bold;
    }
    table.pivot-table th.text-left, table.pivot-table td.text-left { text-align: left; }
    table.pivot-table th.text-right, table.pivot-table td.text-right { text-align: right; }
    table.pivot-table td { 
        padding: 6px 14px; 
        border: 1px solid #555555;     
    }
    table.pivot-table tr:nth-child(even) td { background-color: #F2F5F8; }
    table.pivot-table tr.grand-total-row td { 
        background-color: #D9E1F2; 
        font-weight: bold; 
        color: #000000;
        border-top: 1.5px solid #555555;
        border-bottom: 2px double #555555;
    }
    
    /* 🎯 FIXED: Clean, flat inline link style without structural boxes */
    .mention-tag {
        color: #0066CC;
        font-weight: bold;
        text-decoration: none;
    }
</style>
"""

html_body = f"""
<html>
<head>{html_table_style}</head>
<body>
    <p>Dear GMs and Team,</p>
    <p>Please find attached the list of all open Last Mile (LM) shipments eligible for attempts today.</p>
    
    <p>
        <a class="mention-tag" href="mailto:narasimhareddy.k@flipkart.com">+Narasimha Reddy K</a> 
        <a class="mention-tag" href="mailto:rakhi.kumari@flipkart.com">+Rakhi Kumari</a> 
        <a class="mention-tag" href="mailto:pritam.biswas1@flipkart.com">+Pritam Biswas</a> 
        <a class="mention-tag" href="mailto:renuka.gohil@flipkart.com">+Renuka Gohil</a> 
        — Please ensure all of today's LPD shipments are attempted by EOD (except for cases where the beat is not available).
    </p>
    
    <h3>Table 1: GM vs remark_1 Breakdown (check: Blank)</h3>
    {generate_clean_html_table(pivot_1_sorted, "GM")}
    
    <h3>Table 2: State vs remark_1 Breakdown (check: Blank)</h3>
    {generate_clean_html_table(pivot_2_sorted, "State")}
    
    <h3>Table 3: remark_1 vs LPD Breakdown</h3>
    {generate_clean_html_table(pivot_3_final, "Row Labels")}
    
    <h3>Table 4: Total Volume by remark_1 (check: Blank)</h3>
    {generate_clean_html_table(pivot_4_sorted, "Row Labels")}
    
    <h3>Table 5: Total Volume by Zone</h3>
    {generate_clean_html_table(pivot_5_sorted, "Zone")}
    
    <br>
    <p>Best Regards,<br><strong>Mohammad Ali</strong></p>
</body>
</html>
"""

# ==============================================================================
# 3. SMTP TRANSMISSION MANAGEMENT (UPDATED TO & CC COMPREHENSIVE ROUTING)
# ==============================================================================
sender_email = "mohammadali1.vc@flipkart.com"
sender_password = "wrxe vkic zzik nlye"

# Primary Recipients (TO)
to_list = [
    "vishal.s1@flipkart.com",
    "sapan.agrawal@flipkart.com",
    "hemanth.s@flipkart.com"
    # "mohammadali1.vc@flipkart.com"
]

# Carbon Copy Recipients (CC)
cc_list = [
    "neerajn24.vc@flipkart.com",
    "rakhi.kumari@flipkart.com",
    "harshit.grover@flipkart.com",
    "pritam.biswas1@flipkart.com",
    "rohit.chouhan1@flipkart.com",
    "thimma.reddy@flipkart.com",
    "pravash.singh@flipkart.com",
    "anil.semwal@flipkart.com",
    "deepak.gupta2@flipkart.com",
    "pranjali.kumari@flipkart.com",
    "ramya.thalla@flipkart.com",
    "mrinal.chahal@flipkart.com",
    "carol.dsouza1@flipkart.com",
    "satabdi.ain@flipkart.com",
    "sanjeev.kumar18@flipkart.com",
    "large-ct@flipkart.com",
    "Large-LH@flipkart.com",
    "LargeLH-East@flipkart.com",
    "LargeLH-West@flipkart.com",
    "South_LH@flipkart.com",
    "north_lml_all@flipkart.com",
    "eastlmall@flipkart.com",
    "large-lm-south@flipkart.com",
    "large-cl-central@flipkart.com",
    # "j.praveenkumar@flipkart.com",
    "large-gm@flipkart.com",
    "mohideen.ashraf@flipkart.com",
    "k.dilip@flipkart.com",
    "wahab.b@flipkart.com",
    "vishal.ks@flipkart.com",
    "biplob.dhar@flipkart.com",
    "lm-ops-managers@flipkart.com",
    "edward.cletus@flipkart.com",
    "shahana.c@flipkart.com",
    "prashanth.hn@flipkart.com",
    "gagana.cr@flipkart.com",
    "east_lml@flipkart.com",
    "b.dinesh@flipkart.com",
    "kumar.sai@flipkart.com",
    "dayanidhi.g@flipkart.com",
    "ajay.menon@flipkart.com",
    "rahul.nigam@flipkart.com",
    "girish.singh@flipkart.com",
    "vishal.k@flipkart.com",
    "sujay.sarkar@flipkart.com",
    "krishnakhee.kalita@flipkart.com",
    "praveenm@flipkart.com",
    "mahendran.v@flipkart.com",
    "dey.abhishek@flipkart.com",
    "manishkumar.a@flipkart.com",
    "vipin.sharma2@flipkart.com",
    "client_communications_west@flipkart.com",
    "client_communications_north@flipkart.com",
    "client_communications_south@flipkart.com",
    "client_communications_central@flipkart.com",
    # "rakshamanoja.vc@flipkart.com",
    "attlakumar.vc@flipkart.com",
    "indrajit.mondal@flipkart.com",
    "anunay.m@flipkart.com",
    "renuka.gohil@flipkart.com",
    "narasimhareddy.k@flipkart.com",
    "asrani.kishore@flipkart.com",
    "rakeshkumar.m1@flipkart.com",
    "sankarshan.m@flipkart.com",
    "suranjit.c@flipkart.com",
    "ateequeakhtar.f@flipkart.com",
    "varun.j@flipkart.com",
    "a.deepak@flipkart.com",
    "amit.gupta5@flipkart.com",
    "ujjwal.kumar2@flipkart.com",
    "ajayp3.vc@flipkart.com",
    "abhiram@flipkart.com"
]

# Total delivery mapping array
all_recipients = to_list + cc_list

output_filename = f"Large FWD Pendency {current_date_str}.xlsx" 
smtp_server = "smtp.gmail.com"
smtp_port = 587

msg = MIMEMultipart()
msg['From'] = sender_email

# Set text-headers so Outlook and Gmail format the delivery breakdown beautifully
msg['To'] = ", ".join(to_list)
msg['Cc'] = ", ".join(cc_list)
msg['Subject'] = f"Externalization Large DH pendency, Zero Attempt + NDR of {current_date_str}."

msg.attach(MIMEText(html_body, 'html'))

try:
    with open(output_filename, "rb") as attachment:
        part = MIMEBase("application", "octet-stream")
        part.set_payload(attachment.read())
        encoders.encode_base64(part)
        part.add_header("Content-Disposition", f"attachment; filename={output_filename}")
        msg.attach(part)
except FileNotFoundError:
    print(f"⚠️ Warning: Attachment target '{output_filename}' was not found.")

try:
    server = smtplib.SMTP(smtp_server, smtp_port)
    server.starttls()
    server.login(sender_email, sender_password)
    
    # Sends routing instructions out to the full consolidated list
    server.sendmail(sender_email, all_recipients, msg.as_string())
    server.quit()
    print(f"🚀 Success! Email processed with clean To/CC assignments for all {len(all_recipients)} members.")
except Exception as e:
    print(f"❌ Failed to deliver email: {e}")

🚀 Success! Email processed with clean To/CC assignments for all 73 members.
